# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from collections import Counter
import warnings
import os

warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_fig(name):
    path = os.path.join(OUTPUT_DIR, name)
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  [Saved] {path}")

print("=" * 70)
print("  ROBUST RECOMMENDER SYSTEM – INTERIM ANALYSIS")
print("=" * 70)

  ROBUST RECOMMENDER SYSTEM – INTERIM ANALYSIS


# Data Loading

In [2]:
print("\n[1] LOADING DATA")
print("-" * 40)

movies  = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
users   = pd.read_csv("users.csv")

# Convert timestamp to datetime
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

print(f"  Movies  : {movies.shape[0]:,} rows × {movies.shape[1]} cols")
print(f"  Ratings : {ratings.shape[0]:,} rows × {ratings.shape[1]} cols")
print(f"  Users   : {users.shape[0]:,} rows × {users.shape[1]} cols")


[1] LOADING DATA
----------------------------------------
  Movies  : 3,883 rows × 3 cols
  Ratings : 1,000,209 rows × 4 cols
  Users   : 6,040 rows × 5 cols


# Data Preprocessing

In [3]:
print("\n[2] DATA PREPROCESSING")
print("-" * 40)

# Missing value check 
print("\n  2.1 Missing Value Check")
for name, df in [("movies", movies), ("ratings", ratings), ("users", users)]:
    missing = df.isnull().sum()
    total_missing = missing.sum()
    print(f"    {name:10s}: {total_missing} missing values"
          + (f" → {missing[missing>0].to_dict()}" if total_missing else " ✓"))

# Duplicate detection
print("\n  2.2 Duplicate Detection")
dup_ratings = ratings.duplicated(subset=['UserID','MovieID']).sum()
print(f"    Duplicate (UserID, MovieID) pairs in ratings: {dup_ratings:,}")
if dup_ratings > 0:
    ratings = ratings.drop_duplicates(subset=['UserID','MovieID'], keep='last')
    print(f"    → Duplicates removed. Remaining ratings: {len(ratings):,}")

# Data type casting
print("\n  2.3 Data Type Casting")
ratings['Rating'] = ratings['Rating'].astype(float)
users['Age']      = users['Age'].astype(int)
print("    Rating cast to float, Age to int ✓")

# Genre expansion 
print("\n  2.4 Genre Expansion")
movies['GenreList'] = movies['Genres'].str.split('|')
all_genres = sorted(set(g for gl in movies['GenreList'] for g in gl))
print(f"    Unique genres: {len(all_genres)}")
print(f"    Genre list: {all_genres}")

# Age group mapping 
age_map = {1: "Under 18", 18: "18-24", 25: "25-34", 35: "35-44",
           45: "45-49", 50: "50-55", 56: "56+"}
users['AgeGroup'] = users['Age'].map(age_map)

# Occupation mapping 
occ_map = {
    0: "other/not specified", 1: "academic/educator", 2: "artist",
    3: "clerical/admin", 4: "college/grad student", 5: "customer service",
    6: "doctor/health care", 7: "executive/managerial", 8: "farmer",
    9: "homemaker", 10: "K-12 student", 11: "lawyer", 12: "programmer",
    13: "retired", 14: "sales/marketing", 15: "scientist", 16: "self-employed",
    17: "technician/engineer", 18: "tradesman/craftsman", 19: "unemployed",
    20: "writer"
}
users['OccupationName'] = users['Occupation'].map(occ_map)
print(f"    Age groups and occupation labels mapped ✓")

# Merge into master dataframe
print("\n  2.7 Building Master DataFrame")
master = (ratings
          .merge(users[['UserID','Gender','AgeGroup','OccupationName']], on='UserID')
          .merge(movies[['MovieID','Title','Genres','GenreList']], on='MovieID'))
print(f"    Master shape: {master.shape[0]:,} rows × {master.shape[1]} cols")

# Summary statistics 
print("\n  2.8 Summary Statistics – Ratings")
print(ratings['Rating'].describe().to_string())

print("\n  Data preprocessing complete ✓")


[2] DATA PREPROCESSING
----------------------------------------

  2.1 Missing Value Check
    movies    : 0 missing values ✓
    ratings   : 0 missing values ✓
    users     : 0 missing values ✓

  2.2 Duplicate Detection
    Duplicate (UserID, MovieID) pairs in ratings: 0

  2.3 Data Type Casting
    Rating cast to float, Age to int ✓

  2.4 Genre Expansion
    Unique genres: 18
    Genre list: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
    Age groups and occupation labels mapped ✓

  2.7 Building Master DataFrame
    Master shape: 1,000,209 rows × 10 cols

  2.8 Summary Statistics – Ratings
count    1.000209e+06
mean     3.581564e+00
std      1.117102e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00

  Data preprocessing complete ✓


# Save Merged Master Dataset to CSV

In [4]:
# Save Master DataFrame to CSV 
# After merging all three datasets (ratings + users + movies) into one master
# dataframe, we save it as a CSV file in the current working directory.
# This file serves as the clean ground truth before any noise injection.

print("\n[3] SAVING MERGED MASTER DATASET")
print("-" * 40)

# Drop GenreList (list column — not CSV-friendly) for the saved file
master_csv = master.drop(columns=['GenreList'])

# Save to current working directory
MASTER_CSV_PATH = "master_merged.csv"
master_csv.to_csv(MASTER_CSV_PATH, index=False)

print(f"  master_merged.csv saved ✓")
print(f"  Shape  : {master_csv.shape[0]:,} rows × {master_csv.shape[1]} columns")
print(f"  Columns: {master_csv.columns.tolist()}")
print(f"  Path   : ./{MASTER_CSV_PATH}")
print(f"\n  Preview:")
print(master_csv.head(3).to_string())


[3] SAVING MERGED MASTER DATASET
----------------------------------------
  master_merged.csv saved ✓
  Shape  : 1,000,209 rows × 9 columns
  Columns: ['UserID', 'MovieID', 'Rating', 'Timestamp', 'Gender', 'AgeGroup', 'OccupationName', 'Title', 'Genres']
  Path   : ./master_merged.csv

  Preview:
   UserID  MovieID  Rating           Timestamp Gender  AgeGroup OccupationName                                   Title                        Genres
0       1     1193     5.0 2000-12-31 22:12:40      F  Under 18   K-12 student  One Flew Over the Cuckoo's Nest (1975)                         Drama
1       1      661     3.0 2000-12-31 22:35:09      F  Under 18   K-12 student        James and the Giant Peach (1996)  Animation|Children's|Musical
2       1      914     3.0 2000-12-31 22:32:48      F  Under 18   K-12 student                     My Fair Lady (1964)               Musical|Romance


# Descriptive Analysis and Visualisation

## Rating Distribution

In [5]:
print("\n  3.1 Rating Distribution")
rating_counts = ratings['Rating'].value_counts().sort_index()
print(rating_counts.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(rating_counts.index, rating_counts.values,
              color=sns.color_palette("Blues_d", 5), edgecolor='white', width=0.6)
ax.bar_label(bars, fmt=lambda x: f"{x/1e6:.2f}M", padding=3, fontsize=9)
ax.set_xlabel("Rating Value")
ax.set_ylabel("Number of Ratings")
ax.set_title("Figure 3.1 – Rating Distribution (MovieLens 1M)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
save_fig("fig3_1_rating_distribution.png")


  3.1 Rating Distribution
Rating
1.0     56174
2.0    107557
3.0    261197
4.0    348971
5.0    226310
  [Saved] figures\fig3_1_rating_distribution.png


## Rating Per User (Activity Distribution)

In [6]:
print("\n  3.2 User Activity Distribution")
user_activity = ratings.groupby('UserID')['Rating'].count()
print(f"    Mean ratings/user : {user_activity.mean():.1f}")
print(f"    Median            : {user_activity.median():.0f}")
print(f"    Std               : {user_activity.std():.1f}")
print(f"    Min / Max         : {user_activity.min()} / {user_activity.max():,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_activity, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title("Figure 3.2a – Ratings per User (Full)")
axes[0].set_xlabel("Ratings per User"); axes[0].set_ylabel("Count")

axes[1].hist(user_activity[user_activity < 200], bins=60, color='steelblue', edgecolor='white')
axes[1].set_title("Figure 3.2b – Ratings per User (< 200, zoomed)")
axes[1].set_xlabel("Ratings per User"); axes[1].set_ylabel("Count")
plt.tight_layout()
save_fig("fig3_2_user_activity.png")


  3.2 User Activity Distribution
    Mean ratings/user : 165.6
    Median            : 96
    Std               : 192.7
    Min / Max         : 20 / 2,314
  [Saved] figures\fig3_2_user_activity.png


## Ratings per movie (item popularity)

In [7]:
print("\n  3.3 Item Popularity Distribution")
item_pop = ratings.groupby('MovieID')['Rating'].count()
print(f"    Mean ratings/movie : {item_pop.mean():.1f}")
print(f"    Median             : {item_pop.median():.0f}")
print(f"    Std                : {item_pop.std():.1f}")
print(f"    Min / Max          : {item_pop.min()} / {item_pop.max():,}")

# Long-tail analysis
top_10pct_movies = item_pop.nlargest(int(len(item_pop) * 0.1))
top_10pct_ratings = top_10pct_movies.sum()
print(f"\n    Top 10% movies account for "
      f"{top_10pct_ratings/len(ratings)*100:.1f}% of all ratings  (long-tail effect)")

fig, ax = plt.subplots(figsize=(8, 4))
sorted_pop = item_pop.sort_values(ascending=False).reset_index(drop=True)
ax.plot(sorted_pop.values, color='tomato', linewidth=1.5)
ax.fill_between(range(len(sorted_pop)), sorted_pop.values, alpha=0.15, color='tomato')
ax.set_title("Figure 3.3 – Item Popularity (Long-Tail Distribution)")
ax.set_xlabel("Movie Rank (by popularity)"); ax.set_ylabel("Number of Ratings")
ax.set_yscale('log')
save_fig("fig3_3_item_popularity.png")


  3.3 Item Popularity Distribution
    Mean ratings/movie : 269.9
    Median             : 124
    Std                : 384.0
    Min / Max          : 1 / 3,428

    Top 10% movies account for 44.4% of all ratings  (long-tail effect)
  [Saved] figures\fig3_3_item_popularity.png


## Genre Analysis

In [8]:
print("\n  3.4 Genre Analysis")
genre_counts = Counter(g for gl in movies['GenreList'] for g in gl)
genre_df = pd.DataFrame(genre_counts.items(), columns=['Genre','Count']).sort_values('Count', ascending=False)
print(genre_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=genre_df, x='Count', y='Genre', palette='Blues_r', ax=ax)
ax.set_title("Figure 3.4 – Movie Genre Distribution")
ax.set_xlabel("Number of Movies"); ax.set_ylabel("Genre")
plt.tight_layout()
save_fig("fig3_4_genre_distribution.png")


  3.4 Genre Analysis
      Genre  Count
      Drama   1603
     Comedy   1200
     Action    503
   Thriller    492
    Romance    471
     Horror    343
  Adventure    283
     Sci-Fi    276
 Children's    251
      Crime    211
        War    143
Documentary    127
    Musical    114
    Mystery    106
  Animation    105
    Fantasy     68
    Western     68
  Film-Noir     44
  [Saved] figures\fig3_4_genre_distribution.png


## Average Rating by Genre

In [9]:
print("\n  3.5 Average Rating by Genre")
master_exploded = master.copy()
master_exploded = master_exploded.explode('GenreList')
genre_rating = (master_exploded.groupby('GenreList')['Rating']
                .agg(['mean','count'])
                .rename(columns={'mean':'AvgRating','count':'NumRatings'})
                .sort_values('AvgRating', ascending=False))
print(genre_rating.round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=genre_rating['AvgRating'], y=genre_rating.index, palette='RdYlGn', ax=ax)
ax.set_xlim(3.0, 4.2)
ax.axvline(x=genre_rating['AvgRating'].mean(), color='black', linestyle='--', linewidth=1, label='Mean')
ax.legend()
ax.set_title("Figure 3.5 – Average Rating by Genre")
ax.set_xlabel("Average Rating"); ax.set_ylabel("Genre")
plt.tight_layout()
save_fig("fig3_5_avg_rating_by_genre.png")


  3.5 Average Rating by Genre
             AvgRating  NumRatings
GenreList                         
Film-Noir        4.075       18261
Documentary      3.933        7910
War              3.893       68527
Drama            3.766      354529
Crime            3.709       79541
Animation        3.685       43293
Mystery          3.668       40178
Musical          3.666       41533
Western          3.638       20683
Romance          3.607      147523
Thriller         3.570      189680
Comedy           3.522      356580
Action           3.491      257457
Adventure        3.477      133953
Sci-Fi           3.467      157294
Fantasy          3.447       36301
Children's       3.422       72186
Horror           3.215       76386
  [Saved] figures\fig3_5_avg_rating_by_genre.png


## User Demographics

In [10]:
# User Demographics 
print("\n  3.6 User Demographics")

gender_counts = users['Gender'].value_counts()
age_order     = ["Under 18","18-24","25-34","35-44","45-49","50-55","56+"]
age_counts    = users['AgeGroup'].value_counts().reindex(age_order, fill_value=0)
top_occ       = users['OccupationName'].value_counts().head(10)

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(16, 5))

gender_labels = ['Male' if g == 'M' else 'Female' for g in gender_counts.index]
ax0.pie(gender_counts.values, labels=gender_labels, autopct='%1.1f%%',
        colors=['#4C72B0','#DD8452'], startangle=90)
ax0.set_title("Figure 3.6a – Gender Distribution")

sns.barplot(x=age_counts.values, y=age_counts.index, palette='viridis', ax=ax1)
ax1.set_title("Figure 3.6b – Age Group Distribution")
ax1.set_xlabel("Number of Users"); ax1.set_ylabel("Age Group")

sns.barplot(x=top_occ.values, y=top_occ.index, palette='rocket_r', ax=ax2)
ax2.set_title("Figure 3.6c – Top 10 Occupations")
ax2.set_xlabel("Number of Users"); ax2.set_ylabel("Occupation")

plt.tight_layout()
save_fig("fig3_6_user_demographics.png")


  3.6 User Demographics
  [Saved] figures\fig3_6_user_demographics.png


## Rating Over time

In [11]:
print("\n  3.7 Rating Activity Over Time")
ratings['YearMonth'] = ratings['Timestamp'].dt.to_period('M')
time_activity = ratings.groupby('YearMonth').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_activity.index.astype(str), time_activity.values, color='#2E86AB', linewidth=1.5)
ax.fill_between(range(len(time_activity)), time_activity.values, alpha=0.15, color='#2E86AB')
ax.set_title("Figure 3.7 – Ratings Activity Over Time")
ax.set_xlabel("Year-Month"); ax.set_ylabel("Number of Ratings")
n = max(1, len(time_activity) // 8)
ax.set_xticks(range(0, len(time_activity), n))
ax.set_xticklabels([str(t) for t in time_activity.index[::n]], rotation=45, ha='right')
plt.tight_layout()
save_fig("fig3_7_ratings_over_time.png")


  3.7 Rating Activity Over Time
  [Saved] figures\fig3_7_ratings_over_time.png


# Natural Noise Indicators

## Rating Variance Per User

In [12]:
print("\n[4] NATURAL NOISE INDICATORS")
print("-" * 40)

print("\n  4.1 Rating Variance per User (Inconsistency Indicator)")
user_stats = ratings.groupby('UserID')['Rating'].agg(['mean','std','count'])
user_stats.columns = ['AvgRating','StdRating','NumRatings']
user_stats = user_stats[user_stats['NumRatings'] >= 5]  # min 5 ratings

high_variance_users = (user_stats['StdRating'] > user_stats['StdRating'].quantile(0.90))
print(f"    Total users with ≥5 ratings : {len(user_stats):,}")
print(f"    Mean rating std per user    : {user_stats['StdRating'].mean():.3f}")
print(f"    High-variance users (top 10%): {high_variance_users.sum():,}")
print(f"    → These are natural noise candidates (inconsistent rating behaviour)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_stats['StdRating'].dropna(), bins=50, color='#E76F51', edgecolor='white')
axes[0].axvline(user_stats['StdRating'].quantile(0.90), color='black', linestyle='--', label='90th pct')
axes[0].legend()
axes[0].set_title("Figure 4.1a – Rating Std Dev per User")
axes[0].set_xlabel("Std Dev of Ratings"); axes[0].set_ylabel("Number of Users")

axes[1].scatter(user_stats['NumRatings'], user_stats['StdRating'],
                alpha=0.2, s=5, color='#457B9D')
axes[1].set_title("Figure 4.1b – Rating Activity vs Variance")
axes[1].set_xlabel("Number of Ratings"); axes[1].set_ylabel("Rating Std Dev")
axes[1].set_xscale('log')
plt.tight_layout()
save_fig("fig4_1_user_variance.png")


[4] NATURAL NOISE INDICATORS
----------------------------------------

  4.1 Rating Variance per User (Inconsistency Indicator)
    Total users with ≥5 ratings : 6,040
    Mean rating std per user    : 1.010
    High-variance users (top 10%): 604
    → These are natural noise candidates (inconsistent rating behaviour)
  [Saved] figures\fig4_1_user_variance.png


## Extreme Raters - all 1 or all 5

In [13]:
print("\n  4.2 Extreme Raters (Polar Bimodal Pattern)")
all_high = (user_stats['AvgRating'] >= 4.5) & (user_stats['StdRating'] < 0.5)
all_low  = (user_stats['AvgRating'] <= 1.5) & (user_stats['StdRating'] < 0.5)
print(f"    Always-high raters (avg≥4.5, std<0.5) : {all_high.sum():,}")
print(f"    Always-low  raters (avg≤1.5, std<0.5) : {all_low.sum():,}")
print(f"    → Extreme raters introduce natural noise (lack of discrimination)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_stats['AvgRating'], bins=40, color='#457B9D', edgecolor='white')
ax.axvline(1.5, color='red',   linestyle='--', linewidth=1.5, label='Low threshold (1.5)')
ax.axvline(4.5, color='green', linestyle='--', linewidth=1.5, label='High threshold (4.5)')
ax.set_title("Figure 4.2 – Distribution of Per-User Average Ratings")
ax.set_xlabel("Average Rating per User"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig4_2_extreme_raters.png")


  4.2 Extreme Raters (Polar Bimodal Pattern)
    Always-high raters (avg≥4.5, std<0.5) : 11
    Always-low  raters (avg≤1.5, std<0.5) : 2
    → Extreme raters introduce natural noise (lack of discrimination)
  [Saved] figures\fig4_2_extreme_raters.png


## Rating Entropy Per User

In [14]:
print("\n  4.3 Rating Entropy per User")
def rating_entropy(ratings_series):
    counts = ratings_series.value_counts(normalize=True)
    return -np.sum(counts * np.log2(counts + 1e-10))

user_entropy = ratings.groupby('UserID')['Rating'].apply(rating_entropy)
print(f"    Mean entropy: {user_entropy.mean():.3f} (max possible = log2(5) ≈ 2.32)")
print(f"    Low entropy (<0.5) users: {(user_entropy < 0.5).sum():,}  → narrow rating range (noise risk)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_entropy, bins=40, color='#2A9D8F', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', label='Low-entropy threshold')
ax.set_title("Figure 4.3 – Rating Entropy per User")
ax.set_xlabel("Shannon Entropy of Rating Distribution"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig4_3_rating_entropy.png")


  4.3 Rating Entropy per User
    Mean entropy: 1.853 (max possible = log2(5) ≈ 2.32)
    Low entropy (<0.5) users: 7  → narrow rating range (noise risk)
  [Saved] figures\fig4_3_rating_entropy.png


## Temporal Inconsistancy: Rating Drift

In [15]:
print("\n  4.4 Temporal Rating Drift (Natural Noise)")
# Find users with ≥20 ratings; compute correlation of rating with time
user_drift = []
user_ids_with_enough = ratings.groupby('UserID').filter(lambda x: len(x) >= 20)['UserID'].unique()

np.random.seed(42)
sample_users = np.random.choice(user_ids_with_enough, size=min(500, len(user_ids_with_enough)), replace=False)

for uid in sample_users:
    u_ratings = ratings[ratings['UserID'] == uid].sort_values('Timestamp')
    ts = u_ratings['Timestamp'].astype(np.int64)
    r  = u_ratings['Rating'].values
    if len(r) >= 3 and ts.std() > 0:
        corr, _ = stats.spearmanr(ts, r)
        user_drift.append(corr)

user_drift = np.array(user_drift)
strong_drift = np.abs(user_drift) > 0.3
print(f"    Users sampled for drift analysis : {len(user_drift):,}")
print(f"    Users with |drift correlation| > 0.3 : {strong_drift.sum():,} ({strong_drift.mean()*100:.1f}%)")
print(f"    → Rating drift over time is a key natural noise indicator")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_drift, bins=40, color='#F4A261', edgecolor='white')
ax.axvline(-0.3, color='red', linestyle='--', label='|Drift| > 0.3')
ax.axvline( 0.3, color='red', linestyle='--')
ax.set_title("Figure 4.4 – Temporal Rating Drift (Spearman Corr)")
ax.set_xlabel("Spearman Correlation (Timestamp vs Rating)"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig4_4_temporal_drift.png")


  4.4 Temporal Rating Drift (Natural Noise)
    Users sampled for drift analysis : 500
    Users with |drift correlation| > 0.3 : 88 (17.6%)
    → Rating drift over time is a key natural noise indicator
  [Saved] figures\fig4_4_temporal_drift.png


# Malicious Noise Indicators

## Rating Burst Detection

In [16]:
print("\n  5.1 Rating Burst Detection (Temporal Clustering)")
# Flag users who rate many movies within a short time window (e.g., 1 hour)
ratings_sorted = ratings.sort_values(['UserID', 'Timestamp'])
ratings_sorted['TimeDiff'] = (ratings_sorted.groupby('UserID')['Timestamp']
                               .diff().dt.total_seconds().fillna(9999))

burst_threshold_sec = 3600  # 1 hour
burst_stats = (ratings_sorted[ratings_sorted['TimeDiff'] < burst_threshold_sec]
               .groupby('UserID').size()
               .rename('BurstRatings'))

users_with_bursts = (burst_stats > 10)  # >10 rapid ratings
print(f"    Users with >10 rapid ratings (<1hr gap): {users_with_bursts.sum():,}")
print(f"    → Rapid-fire rating bursts are a shilling attack signal")

fig, ax = plt.subplots(figsize=(7, 4))
burst_values = burst_stats.values
ax.hist(burst_values[burst_values < 100], bins=50, color='#E63946', edgecolor='white')
ax.axvline(10, color='black', linestyle='--', label='Burst threshold (10)')
ax.set_title("Figure 5.1 – Rating Burst Counts per User")
ax.set_xlabel("Number of Rapid Successive Ratings"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig5_1_rating_bursts.png")


  5.1 Rating Burst Detection (Temporal Clustering)
    Users with >10 rapid ratings (<1hr gap): 6,040
    → Rapid-fire rating bursts are a shilling attack signal
  [Saved] figures\fig5_1_rating_bursts.png


## Filler Item Detection

In [17]:
print("\n  5.2 Filler Item Ratings (Popular Item Coverage)")
popular_movies = item_pop[item_pop > item_pop.quantile(0.80)].index
filler_ratio = (ratings[ratings['MovieID'].isin(popular_movies)]
                .groupby('UserID').size() /
                ratings.groupby('UserID').size())
filler_ratio = filler_ratio.dropna()

high_filler = filler_ratio > 0.8
print(f"    Users with >80% of ratings on popular (top 20%) movies: {high_filler.sum():,}")
print(f"    → High filler ratio → shilling indicator (disguise via popular items)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(filler_ratio, bins=50, color='#6A0572', edgecolor='white', alpha=0.8)
ax.axvline(0.8, color='red', linestyle='--', label='>80% filler threshold')
ax.set_title("Figure 5.2 – Filler Item Ratio per User")
ax.set_xlabel("Fraction of Ratings on Popular (Top-20%) Movies"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig5_2_filler_ratio.png")


  5.2 Filler Item Ratings (Popular Item Coverage)
    Users with >80% of ratings on popular (top 20%) movies: 1,709
    → High filler ratio → shilling indicator (disguise via popular items)
  [Saved] figures\fig5_2_filler_ratio.png


## User Rating Deviation From Mean

In [18]:
print("\n  5.3 User Mean Rating Deviation from Global Mean")
global_mean = ratings['Rating'].mean()
user_mean_dev = np.abs(user_stats['AvgRating'] - global_mean)
high_dev = user_mean_dev > 1.5
print(f"    Global mean rating                    : {global_mean:.3f}")
print(f"    Users with |avg_rating - global| > 1.5: {high_dev.sum():,}")
print(f"    → Large deviations indicate biased/manipulative rating patterns")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_mean_dev, bins=50, color='#F77F00', edgecolor='white')
ax.axvline(1.5, color='red', linestyle='--', label='|Deviation| > 1.5')
ax.set_title("Figure 5.3 – Per-User Mean Rating Deviation from Global Mean")
ax.set_xlabel("Absolute Deviation"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig5_3_rating_deviation.png")


  5.3 User Mean Rating Deviation from Global Mean
    Global mean rating                    : 3.582
    Users with |avg_rating - global| > 1.5: 10
    → Large deviations indicate biased/manipulative rating patterns
  [Saved] figures\fig5_3_rating_deviation.png


## Composite Shalling Suspicion Score

In [19]:
print("\n  5.4 Composite Shilling Suspicion Score")
suspicion = pd.DataFrame(index=user_stats.index)
suspicion['HighFiller']   = (filler_ratio > 0.8).astype(int)
suspicion['LowVariance']  = (user_stats['StdRating'] < 0.3).astype(int)
suspicion['HighDeviation']= (user_mean_dev > 1.5).astype(int)
burst_flag = (burst_stats > 10).reindex(user_stats.index, fill_value=0).astype(int)
suspicion['BurstRating']  = burst_flag
suspicion['SuspicionScore'] = suspicion.sum(axis=1)

score_dist = suspicion['SuspicionScore'].value_counts().sort_index()
print(score_dist.to_string())
high_risk = (suspicion['SuspicionScore'] >= 3)
print(f"\n    High-risk users (score ≥ 3): {high_risk.sum():,} "
      f"({high_risk.mean()*100:.2f}% of users)")

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2DC653','#B0E0A8','#FCBA03','#FC6F03','#D00000'][:len(score_dist)]
ax.bar(score_dist.index, score_dist.values, color=colors, edgecolor='white', width=0.6)
ax.set_title("Figure 5.4 – Composite Shilling Suspicion Score Distribution")
ax.set_xlabel("Suspicion Score (0–4)"); ax.set_ylabel("Number of Users")
for i, (idx, val) in enumerate(score_dist.items()):
    ax.text(idx, val + 20, str(val), ha='center', fontsize=9)
save_fig("fig5_4_suspicion_score.png")


  5.4 Composite Shilling Suspicion Score
SuspicionScore
1    4319
2    1720
3       1

    High-risk users (score ≥ 3): 1 (0.02% of users)
  [Saved] figures\fig5_4_suspicion_score.png


# Sparsity Analysis

In [20]:
print("\n[6] SPARSITY ANALYSIS")
print("-" * 40)

n_users  = ratings['UserID'].nunique()
n_movies = ratings['MovieID'].nunique()
n_ratings = len(ratings)
density  = n_ratings / (n_users * n_movies) * 100
sparsity = 100 - density

print(f"    Users         : {n_users:,}")
print(f"    Movies        : {n_movies:,}")
print(f"    Ratings       : {n_ratings:,}")
print(f"    Matrix size   : {n_users:,} × {n_movies:,} = {n_users*n_movies:,}")
print(f"    Density       : {density:.4f}%")
print(f"    Sparsity      : {sparsity:.4f}%")
print(f"    → High sparsity amplifies both natural and malicious noise effects")


[6] SPARSITY ANALYSIS
----------------------------------------
    Users         : 6,040
    Movies        : 3,706
    Ratings       : 1,000,209
    Matrix size   : 6,040 × 3,706 = 22,384,240
    Density       : 4.4684%
    Sparsity      : 95.5316%
    → High sparsity amplifies both natural and malicious noise effects


# Cold-Start Analysis

In [21]:
cold_start_users  = (ratings.groupby('UserID').size() < 10).sum()
cold_start_movies = (ratings.groupby('MovieID').size() < 10).sum()
print(f"\n    Cold-start users  (< 10 ratings) : {cold_start_users:,}")
print(f"    Cold-start movies (< 10 ratings) : {cold_start_movies:,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
user_counts = ratings.groupby('UserID').size()
movie_counts = ratings.groupby('MovieID').size()

axes[0].hist(user_counts, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(10, color='red', linestyle='--', label='Cold-start threshold (10)')
axes[0].set_title("Figure 6a – User Interaction Counts")
axes[0].set_xlabel("Ratings per User"); axes[0].set_ylabel("Count"); axes[0].legend()

axes[1].hist(movie_counts, bins=60, color='tomato', edgecolor='white')
axes[1].axvline(10, color='black', linestyle='--', label='Cold-start threshold (10)')
axes[1].set_title("Figure 6b – Movie Interaction Counts")
axes[1].set_xlabel("Ratings per Movie"); axes[1].set_ylabel("Count"); axes[1].legend()
plt.tight_layout()
save_fig("fig6_sparsity.png")


    Cold-start users  (< 10 ratings) : 0
    Cold-start movies (< 10 ratings) : 446
  [Saved] figures\fig6_sparsity.png


# Conceptual Analaysis - Noise Taxonomy

## Noise Indicator Correalation Matrix

In [22]:
print("\n  7.1 Noise Feature Correlation Matrix")
noise_features = pd.DataFrame({
    'AvgRating'      : user_stats['AvgRating'],
    'StdRating'      : user_stats['StdRating'],
    'NumRatings'     : user_stats['NumRatings'],
    'MeanDeviation'  : user_mean_dev,
    'FillerRatio'    : filler_ratio.reindex(user_stats.index),
    'SuspicionScore' : suspicion['SuspicionScore'],
}).dropna()

corr_matrix = noise_features.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title("Figure 7.1 – Noise Feature Correlation Heatmap")
plt.tight_layout()
save_fig("fig7_1_correlation_heatmap.png")


  7.1 Noise Feature Correlation Matrix
  [Saved] figures\fig7_1_correlation_heatmap.png


## High Suspicion vs Normal User Comparison

In [23]:
print("\n  7.2 High-Suspicion vs Normal Users – Feature Comparison")
noise_features['IsHighRisk'] = (suspicion['SuspicionScore'] >= 2).reindex(noise_features.index, fill_value=False)

comparison = noise_features.groupby('IsHighRisk')[
    ['AvgRating','StdRating','NumRatings','FillerRatio']].mean().round(3)
comparison.index = ['Normal Users', 'High-Risk Users']
print(comparison.to_string())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
features = ['AvgRating','StdRating','NumRatings','FillerRatio']
labels   = ['Avg Rating','Rating Std Dev','Num Ratings','Filler Ratio']
for i, (feat, lab) in enumerate(zip(features, labels)):
    vals = [noise_features[noise_features['IsHighRisk']==False][feat].mean(),
            noise_features[noise_features['IsHighRisk']==True ][feat].mean()]
    axes[i].bar(['Normal','High-Risk'], vals,
                color=['#2DC653','#D00000'], edgecolor='white', width=0.5)
    axes[i].set_title(f"Figure 7.2 – {lab}")
    axes[i].set_ylabel(lab)
plt.suptitle("Figure 7.2 – Feature Comparison: Normal vs High-Risk Users", y=1.02)
plt.tight_layout()
save_fig("fig7_2_user_comparison.png")


  7.2 High-Suspicion vs Normal Users – Feature Comparison
                 AvgRating  StdRating  NumRatings  FillerRatio
Normal Users         3.655      1.026     199.714        0.656
High-Risk Users      3.822      0.972      79.979        0.861
  [Saved] figures\fig7_2_user_comparison.png


# Final Noise Classification Summary

In [24]:
print("\n  7.3 Noise Classification Summary")
print("""
  ┌──────────────────────────────────────────────────────────────────────┐
  │             NOISE TAXONOMY IN MOVIELENS 1M DATASET                  │
  ├──────────────────────┬───────────────────────────────────────────────┤
  │  NATURAL NOISE       │  Indicators Found                            │
  │  (Benign)            │  • High rating variance per user             │
  │                      │  • Extreme raters (all-1s or all-5s)         │
  │                      │  • Low entropy users (narrow rating spread)  │
  │                      │  • Temporal rating drift (mood/context shift)│
  ├──────────────────────┼───────────────────────────────────────────────┤
  │  MALICIOUS NOISE     │  Proxy Indicators Found                      │
  │  (Shilling Attacks)  │  • Rating bursts (rapid successive ratings)  │
  │                      │  • High filler item ratio (popular coverage) │
  │                      │  • Extreme deviation from global mean        │
  │                      │  • Composite high-suspicion score            │
  ├──────────────────────┼───────────────────────────────────────────────┤
  │  DATA QUALITY        │  • Sparsity: ~95.53% missing entries         │
  │  CONCERNS            │  • Cold-start problem present                │
  │                      │  • Long-tail item popularity distribution    │
  └──────────────────────┴───────────────────────────────────────────────┘
""")


  7.3 Noise Classification Summary

  ┌──────────────────────────────────────────────────────────────────────┐
  │             NOISE TAXONOMY IN MOVIELENS 1M DATASET                  │
  ├──────────────────────┬───────────────────────────────────────────────┤
  │  NATURAL NOISE       │  Indicators Found                            │
  │  (Benign)            │  • High rating variance per user             │
  │                      │  • Extreme raters (all-1s or all-5s)         │
  │                      │  • Low entropy users (narrow rating spread)  │
  │                      │  • Temporal rating drift (mood/context shift)│
  ├──────────────────────┼───────────────────────────────────────────────┤
  │  MALICIOUS NOISE     │  Proxy Indicators Found                      │
  │  (Shilling Attacks)  │  • Rating bursts (rapid successive ratings)  │
  │                      │  • High filler item ratio (popular coverage) │
  │                      │  • Extreme deviation from global mean        

# Final Summary Table

In [25]:
print("\n[8] DATASET SUMMARY REPORT")
print("=" * 70)
summary = {
    "Total Ratings"                    : f"{n_ratings:,}",
    "Unique Users"                     : f"{n_users:,}",
    "Unique Movies"                    : f"{n_movies:,}",
    "Rating Scale"                     : "1–5 (integer)",
    "Matrix Density"                   : f"{density:.4f}%",
    "Matrix Sparsity"                  : f"{sparsity:.4f}%",
    "Mean Ratings/User"                : f"{user_activity.mean():.1f}",
    "Mean Ratings/Movie"               : f"{item_pop.mean():.1f}",
    "High-Variance Users (top 10%)"    : f"{high_variance_users.sum():,}",
    "Extreme Raters (all-high/all-low)": f"{all_high.sum() + all_low.sum():,}",
    "Burst Raters (>10 rapid)"         : f"{users_with_bursts.sum():,}",
    "High Filler Ratio Users (>80%)"   : f"{high_filler.sum():,}",
    "High-Risk Users (score≥3)"        : f"{high_risk.sum():,}",
    "Cold-Start Users (<10 ratings)"   : f"{cold_start_users:,}",
    "Cold-Start Movies (<10 ratings)"  : f"{cold_start_movies:,}",
}
for k, v in summary.items():
    print(f"  {k:<42}: {v}")

print("\n" + "=" * 70)
print("  All figures saved to ./figures/")
print("  Analysis complete ✓")
print("=" * 70)


[8] DATASET SUMMARY REPORT
  Total Ratings                             : 1,000,209
  Unique Users                              : 6,040
  Unique Movies                             : 3,706
  Rating Scale                              : 1–5 (integer)
  Matrix Density                            : 4.4684%
  Matrix Sparsity                           : 95.5316%
  Mean Ratings/User                         : 165.6
  Mean Ratings/Movie                        : 269.9
  High-Variance Users (top 10%)             : 604
  Extreme Raters (all-high/all-low)         : 13
  Burst Raters (>10 rapid)                  : 6,040
  High Filler Ratio Users (>80%)            : 1,709
  High-Risk Users (score≥3)                 : 1
  Cold-Start Users (<10 ratings)            : 0
  Cold-Start Movies (<10 ratings)           : 446

  All figures saved to ./figures/
  Analysis complete ✓


---
# Malicious Noise Injection
## Research Context
The MovieLens 1M dataset is a clean academic benchmark — it contains no confirmed malicious users.  
To evaluate the UNRR framework's ability to detect and mitigate shilling attacks, we inject **four standard attack types** into a 100K sample of the merged master dataset.

| Attack Type | Strategy | Goal |
|---|---|---|
| **Random Attack** | Random filler ratings + max score on target | Simple push attack |
| **Average Attack** | Item-mean filler ratings + max score on target | Hard-to-detect push |
| **Bandwagon Attack** | Popular item filler + max score on target | Exploit popularity bias |
| **Love/Hate Attack** | All 5s (push) or all 1s (nuke) | Extreme rating manipulation |

## Configuration & Setup

In [26]:
# Malicious Noise Injection — Configuration 
import random

INJECTION_CONFIG = {
    'sample_size'         : 100_000,   # Ratings to sample from master
    'random_seed'         : 42,        # Reproducibility
    'attack_percentage'   : 0.05,      # 5% of users will be fake attackers
    'filler_size'         : 50,        # Filler items rated per attacker
    'target_size'         : 5,         # Target items per attacker (push goal)
    'push_ratio'          : 0.5,       # 50% push / 50% nuke for love-hate
}

random.seed(INJECTION_CONFIG['random_seed'])
np.random.seed(INJECTION_CONFIG['random_seed'])

INJECT_OUTPUT_DIR = "injection_output"
os.makedirs(INJECT_OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("  MALICIOUS NOISE INJECTION")
print("  Robust Recommender System — Group 17, CDU")
print("=" * 60)
print(f"\n  Configuration:")
for k, v in INJECTION_CONFIG.items():
    print(f"    {k:<25}: {v}")

  MALICIOUS NOISE INJECTION
  Robust Recommender System — Group 17, CDU

  Configuration:
    sample_size              : 100000
    random_seed              : 42
    attack_percentage        : 0.05
    filler_size              : 50
    target_size              : 5
    push_ratio               : 0.5


## Step 1 — Load Merged CSV & Sample 100K

In [27]:
# Load master_merged.csv and sample 100K 
# We load the merged CSV saved earlier (master_merged.csv) as the base dataset.
# A 100K sample is taken for injection — manageable size for framework testing.

print("\n[INJECT-1] Loading master_merged.csv and sampling 100K...")

master_full = pd.read_csv("master_merged.csv")

# Sample 100K rows reproducibly
ml100k = master_full.sample(
    n=INJECTION_CONFIG['sample_size'],
    random_state=INJECTION_CONFIG['random_seed']
).reset_index(drop=True)

# Save clean 100K original as ground truth
original_path = os.path.join(INJECT_OUTPUT_DIR, "ml100k_original.csv")
ml100k.to_csv(original_path, index=False)

print(f"  master_merged.csv loaded  : {master_full.shape[0]:,} rows")
print(f"  100K sample created       : {len(ml100k):,} rows")
print(f"  Unique users              : {ml100k['UserID'].nunique():,}")
print(f"  Unique movies             : {ml100k['MovieID'].nunique():,}")
print(f"  Global mean rating        : {ml100k['Rating'].mean():.4f}")
print(f"  Clean original saved  → {original_path}")
print(f"\n  Columns available: {ml100k.columns.tolist()}")


[INJECT-1] Loading master_merged.csv and sampling 100K...
  master_merged.csv loaded  : 1,000,209 rows
  100K sample created       : 100,000 rows
  Unique users              : 5,970
  Unique movies             : 3,294
  Global mean rating        : 3.5826
  Clean original saved  → injection_output\ml100k_original.csv

  Columns available: ['UserID', 'MovieID', 'Rating', 'Timestamp', 'Gender', 'AgeGroup', 'OccupationName', 'Title', 'Genres']


## Step 2 — Prepare Attack Parameters

In [28]:
# Prepare attack parameters 
# Identify popular items (for bandwagon filler), target items (least-popular,
# simulate push attack on niche movies), and attacker count per type.

print("\n[INJECT-2] Preparing attack parameters...")

n_users_100k    = ml100k['UserID'].nunique()
global_mean_100k = ml100k['Rating'].mean()

# Item popularity in the 100K sample
item_pop_100k   = ml100k.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
all_items_100k  = ml100k['MovieID'].unique().tolist()
popular_items_100k = item_pop_100k.head(100).index.tolist()   # top 100 popular

# Target items — least popular movies (simulate push attack on niche items)
target_items_100k = item_pop_100k.tail(50).index.tolist()
target_items_100k = random.sample(
    target_items_100k,
    min(INJECTION_CONFIG['target_size'], len(target_items_100k))
)

# Attacker counts
n_attackers  = max(1, int(n_users_100k * INJECTION_CONFIG['attack_percentage']))
n_random     = max(1, int(n_attackers * 0.25))
n_average    = max(1, int(n_attackers * 0.25))
n_bandwagon  = max(1, int(n_attackers * 0.25))
n_lovehate   = n_attackers - n_random - n_average - n_bandwagon

# Fake UserIDs start above the maximum real UserID
fake_user_start = ml100k['UserID'].max() + 1

# Timestamp range for realistic fake timestamps
ts_series   = pd.to_datetime(ml100k['Timestamp']).astype(np.int64) // 10**9
ts_min_unix = int(ts_series.min())
ts_max_unix = int(ts_series.max())

print(f"  Real users in 100K        : {n_users_100k:,}")
print(f"  Total fake attackers      : {n_attackers} ({INJECTION_CONFIG['attack_percentage']*100:.0f}% of users)")
print(f"    Random Attack           : {n_random}")
print(f"    Average Attack          : {n_average}")
print(f"    Bandwagon Attack        : {n_bandwagon}")
print(f"    Love/Hate Attack        : {n_lovehate}")
print(f"  Target movies (push goal) : {target_items_100k}")
print(f"  Filler size per attacker  : {INJECTION_CONFIG['filler_size']}")
print(f"  Fake UserIDs start at     : {fake_user_start}")


[INJECT-2] Preparing attack parameters...
  Real users in 100K        : 5,970
  Total fake attackers      : 298 (5% of users)
    Random Attack           : 74
    Average Attack          : 74
    Bandwagon Attack        : 74
    Love/Hate Attack        : 76
  Target movies (push goal) : [397, 1504, 3116, 2977, 2063]
  Filler size per attacker  : 50
  Fake UserIDs start at     : 6041


## Step 3 — Attack Generation Functions

In [29]:
# Attack generation helper 
def burst_timestamp():
    """All ratings within a 1-hour burst window — shilling temporal signature."""
    start = random.randint(ts_min_unix, ts_max_unix - 3600)
    return start

def make_profile(user_id, items, ratings_list, attack_type, is_filler_list, is_target_list, burst_ts):
    """Build a list of rating dicts for one fake user."""
    rows = []
    for item, rating, is_filler, is_target in zip(items, ratings_list, is_filler_list, is_target_list):
        rows.append({
            'UserID'      : user_id,
            'MovieID'     : item,
            'Rating'      : float(rating),
            'Timestamp'   : burst_ts + random.randint(0, 3600),
            'AttackType'  : attack_type,
            'IsFiller'    : is_filler,
            'IsTarget'    : is_target,
            'IsMalicious' : True
        })
    return rows


# ── ATTACK TYPE 1: Random Attack ──────────────────────────────────────────────
def random_attack(user_id):
    """
    Filler items get random ratings (1-5) to mimic normal user behaviour.
    Target items get maximum rating (5) to push them up the recommendation list.
    Easiest attack to detect due to random filler rating variance.
    """
    filler = random.sample([i for i in all_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(all_items_100k)-len(target_items_100k)))
    ts = burst_timestamp()
    items   = filler + target_items_100k
    ratings_vals = [float(random.randint(1,5)) for _ in filler] + [5.0]*len(target_items_100k)
    fillers = [True]*len(filler) + [False]*len(target_items_100k)
    targets = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, 'Random', fillers, targets, ts)


# ── ATTACK TYPE 2: Average Attack ─────────────────────────────────────────────
def average_attack(user_id):
    """
    Filler items rated near their item mean with small Gaussian noise.
    This makes the attacker look like a genuine user — harder to detect.
    Target items still receive maximum rating (5).
    """
    filler = random.sample([i for i in all_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(all_items_100k)-len(target_items_100k)))
    item_means = ml100k.groupby('MovieID')['Rating'].mean()
    ts = burst_timestamp()
    filler_ratings = []
    for item in filler:
        mean_r = item_means.get(item, global_mean_100k)
        r = round(float(np.clip(mean_r + np.random.normal(0, 0.3), 1, 5)) * 2) / 2
        filler_ratings.append(r)
    items   = filler + target_items_100k
    ratings_vals = filler_ratings + [5.0]*len(target_items_100k)
    fillers = [True]*len(filler) + [False]*len(target_items_100k)
    targets = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, 'Average', fillers, targets, ts)


# ── ATTACK TYPE 3: Bandwagon Attack ───────────────────────────────────────────
def bandwagon_attack(user_id):
    """
    Filler items are drawn exclusively from the top-100 popular movies.
    High ratings on popular items make the profile appear genuine and engaged.
    Most difficult to detect — exploits natural popularity bias in the system.
    """
    filler = random.sample([i for i in popular_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(popular_items_100k)))
    ts = burst_timestamp()
    filler_ratings = [float(random.choice([3,4,4,5,5])) for _ in filler]
    items   = filler + target_items_100k
    ratings_vals = filler_ratings + [5.0]*len(target_items_100k)
    fillers = [True]*len(filler) + [False]*len(target_items_100k)
    targets = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, 'Bandwagon', fillers, targets, ts)


# ── ATTACK TYPE 4: Love/Hate Attack ───────────────────────────────────────────
def lovehate_attack(user_id, is_push=True):
    """
    Push (Love): All ratings are 5 — artificially inflate target item scores.
    Nuke (Hate): All ratings are 1 — artificially suppress competitor items.
    Simplest attack — zero rating variance makes it the easiest to detect.
    """
    filler = random.sample([i for i in all_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(all_items_100k)-len(target_items_100k)))
    ts         = burst_timestamp()
    r_val      = 5.0 if is_push else 1.0
    label      = 'Love-Push' if is_push else 'Hate-Nuke'
    items      = filler + target_items_100k
    ratings_vals = [r_val]*len(items)
    fillers    = [True]*len(filler) + [False]*len(target_items_100k)
    targets    = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, label, fillers, targets, ts)

print("  All 4 attack functions defined ✓")
print("  Attack Type 1 : Random Attack")
print("  Attack Type 2 : Average Attack")
print("  Attack Type 3 : Bandwagon Attack")
print("  Attack Type 4 : Love/Hate Attack (Push + Nuke)")

  All 4 attack functions defined ✓
  Attack Type 1 : Random Attack
  Attack Type 2 : Average Attack
  Attack Type 3 : Bandwagon Attack
  Attack Type 4 : Love/Hate Attack (Push + Nuke)


## Step 4 — Generate & Inject All Fake Profiles

In [30]:
# Generate all fake profiles and inject into 100K dataset 
print("\n[INJECT-3] Generating malicious profiles...")

all_fake = []
uid = fake_user_start

# Random attackers
for _ in range(n_random):
    all_fake.extend(random_attack(uid)); uid += 1

# Average attackers
for _ in range(n_average):
    all_fake.extend(average_attack(uid)); uid += 1

# Bandwagon attackers
for _ in range(n_bandwagon):
    all_fake.extend(bandwagon_attack(uid)); uid += 1

# Love/Hate attackers (split push/nuke)
n_push = int(n_lovehate * INJECTION_CONFIG['push_ratio'])
for i in range(n_lovehate):
    all_fake.extend(lovehate_attack(uid, is_push=(i < n_push))); uid += 1

fake_df = pd.DataFrame(all_fake)

print(f"  Fake attacker profiles    : {n_attackers}")
print(f"  Total fake ratings        : {len(fake_df):,}")
print(f"\n  Breakdown by attack type:")
print(fake_df.groupby('AttackType').agg(
    Profiles=('UserID','nunique'),
    Ratings=('Rating','count'),
    AvgRating=('Rating','mean')
).round(3).to_string())

# ── Label original 100K with noise columns ───────────────────────────────────
ml100k_labeled = ml100k.copy()
ml100k_labeled['AttackType']  = 'None'
ml100k_labeled['IsFiller']    = False
ml100k_labeled['IsTarget']    = False
ml100k_labeled['IsMalicious'] = False

# ── Combine real + fake into injected dataset ─────────────────────────────────
# Align columns — fake_df has subset of columns from master
fake_aligned = pd.DataFrame({
    'UserID'      : fake_df['UserID'],
    'MovieID'     : fake_df['MovieID'],
    'Rating'      : fake_df['Rating'],
    'Timestamp'   : pd.to_datetime(fake_df['Timestamp'], unit='s'),
    'Gender'      : 'U',           # Unknown — fake user
    'AgeGroup'    : 'Unknown',
    'OccupationName': 'Unknown',
    'Title'       : 'Unknown',
    'Genres'      : 'Unknown',
    'AttackType'  : fake_df['AttackType'],
    'IsFiller'    : fake_df['IsFiller'],
    'IsTarget'    : fake_df['IsTarget'],
    'IsMalicious' : fake_df['IsMalicious'],
})

injected_df = pd.concat([ml100k_labeled, fake_aligned], ignore_index=True)
injected_df = injected_df.sort_values(['UserID','Timestamp']).reset_index(drop=True)

# ── Save output files ─────────────────────────────────────────────────────────
injected_path = os.path.join(INJECT_OUTPUT_DIR, "ml100k_injected.csv")
fake_path     = os.path.join(INJECT_OUTPUT_DIR, "malicious_profiles.csv")

injected_df.to_csv(injected_path, index=False)
fake_df.to_csv(fake_path, index=False)

contamination = len(fake_df) / len(injected_df) * 100
print(f"\n[INJECT-4] Injection complete")
print(f"  Original 100K ratings     : {len(ml100k):,}")
print(f"  Injected fake ratings     : {len(fake_df):,}")
print(f"  Total injected dataset    : {len(injected_df):,}")
print(f"  Contamination rate        : {contamination:.2f}%")
print(f"\n  Saved → {injected_path}")
print(f"  Saved → {fake_path}")


[INJECT-3] Generating malicious profiles...
  Fake attacker profiles    : 298
  Total fake ratings        : 16,390

  Breakdown by attack type:
            Profiles  Ratings  AvgRating
AttackType                              
Average           74     4070      3.444
Bandwagon         74     4070      4.268
Hate-Nuke         38     2090      1.000
Love-Push         38     2090      5.000
Random            74     4070      3.242

[INJECT-4] Injection complete
  Original 100K ratings     : 100,000
  Injected fake ratings     : 16,390
  Total injected dataset    : 116,390
  Contamination rate        : 14.08%

  Saved → injection_output\ml100k_injected.csv
  Saved → injection_output\malicious_profiles.csv


## Step 5 — Visualise Attack Impact

In [31]:
# Figure: Rating distribution before vs after injection
print("\n[INJECT-5] Visualising attack impact...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Rating distribution comparison
rating_vals = [1.0, 2.0, 3.0, 4.0, 5.0]
orig_counts = [( ml100k['Rating'] == r).sum() for r in rating_vals]
inj_counts  = [(injected_df['Rating'] == r).sum() for r in rating_vals]

x = np.arange(len(rating_vals))
w = 0.35
axes[0].bar(x - w/2, orig_counts, w, label='Original', color='#4C72B0', edgecolor='white')
axes[0].bar(x + w/2, inj_counts,  w, label='Injected',  color='#D85A30', edgecolor='white')
axes[0].set_title("Fig I.1 – Rating Distribution: Before vs After Injection")
axes[0].set_xlabel("Rating Value"); axes[0].set_ylabel("Count")
axes[0].set_xticks(x); axes[0].set_xticklabels(['1','2','3','4','5'])
axes[0].legend()

# Plot 2: Attack type breakdown
atk_counts = fake_df.groupby('AttackType')['Rating'].count()
colors_atk = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']
axes[1].bar(atk_counts.index, atk_counts.values,
            color=colors_atk[:len(atk_counts)], edgecolor='white')
axes[1].set_title("Fig I.2 – Fake Ratings by Attack Type")
axes[1].set_xlabel("Attack Type"); axes[1].set_ylabel("Number of Fake Ratings")
axes[1].tick_params(axis='x', rotation=20)
for i, (idx, val) in enumerate(atk_counts.items()):
    axes[1].text(i, val + 10, str(val), ha='center', fontsize=9)

# Plot 3: Target item rating impact
target_before = [ml100k[ml100k['MovieID']==t]['Rating'].mean() for t in target_items_100k]
target_after  = [injected_df[injected_df['MovieID']==t]['Rating'].mean() for t in target_items_100k]
x_t = np.arange(len(target_items_100k))
axes[2].bar(x_t - w/2, target_before, w, label='Before', color='#4C72B0', edgecolor='white')
axes[2].bar(x_t + w/2, target_after,  w, label='After',  color='#D85A30', edgecolor='white')
axes[2].set_title("Fig I.3 – Target Item Avg Rating: Before vs After Attack")
axes[2].set_xlabel("Target Movie ID"); axes[2].set_ylabel("Average Rating")
axes[2].set_xticks(x_t)
axes[2].set_xticklabels([str(t) for t in target_items_100k], rotation=15)
axes[2].set_ylim(0, 5.5); axes[2].legend()

plt.tight_layout()
save_fig("fig_injection_impact.png")

# Print rating mean shift
print(f"  Original mean rating : {ml100k['Rating'].mean():.4f}")
print(f"  Injected mean rating : {injected_df['Rating'].mean():.4f}")
print(f"  Bias introduced      : {injected_df['Rating'].mean() - ml100k['Rating'].mean():+.4f}")


[INJECT-5] Visualising attack impact...
  [Saved] figures\fig_injection_impact.png
  Original mean rating : 3.5826
  Injected mean rating : 3.5689
  Bias introduced      : -0.0137


## Step 6 — Injection Summary Report

In [32]:
# Final injection summary 
print("\n" + "=" * 60)
print("  MALICIOUS NOISE INJECTION — SUMMARY REPORT")
print("=" * 60)

print(f"""
  DATASET
  -------
  Base dataset        : master_merged.csv (full 1M merge)
  Sample size         : {len(ml100k):,} ratings
  Real users          : {ml100k['UserID'].nunique():,}
  Real movies         : {ml100k['MovieID'].nunique():,}

  INJECTION
  ---------
  Total fake profiles : {n_attackers}
    Random Attack     : {n_random}
    Average Attack    : {n_average}
    Bandwagon Attack  : {n_bandwagon}
    Love/Hate Attack  : {n_lovehate}
  Total fake ratings  : {len(fake_df):,}
  Contamination rate  : {contamination:.2f}%
  Fake UserIDs        : {fake_user_start} – {uid-1}

  TARGET ITEMS (Push Attack Goal)
  --------------------------------""")

for t in target_items_100k:
    b = ml100k[ml100k['MovieID']==t]['Rating'].mean()
    a = injected_df[injected_df['MovieID']==t]['Rating'].mean()
    n = fake_df[fake_df['MovieID']==t].shape[0]
    print(f"  MovieID {t:<8}: {b:.3f} → {a:.3f}  (change: {a-b:+.3f}, {n} fake ratings)")

print(f"""
  OUTPUT FILES
  ------------
  injection_output/ml100k_original.csv    → Clean 100K ground truth
  injection_output/ml100k_injected.csv    → Contaminated dataset (use for UNRR testing)
  injection_output/malicious_profiles.csv → Fake profiles only

  KEY COLUMN: IsMalicious
  -----------------------
  False → genuine user rating
  True  → injected fake rating  (ground truth label for UNRR evaluation)

  AttackType column values:
  None / Random / Average / Bandwagon / Love-Push / Hate-Nuke
""")
print("=" * 60)
print("  Injection complete ✓  Ready for UNRR framework evaluation")
print("=" * 60)


  MALICIOUS NOISE INJECTION — SUMMARY REPORT

  DATASET
  -------
  Base dataset        : master_merged.csv (full 1M merge)
  Sample size         : 100,000 ratings
  Real users          : 5,970
  Real movies         : 3,294

  INJECTION
  ---------
  Total fake profiles : 298
    Random Attack     : 74
    Average Attack    : 74
    Bandwagon Attack  : 74
    Love/Hate Attack  : 76
  Total fake ratings  : 16,390
  Contamination rate  : 14.08%
  Fake UserIDs        : 6041 – 6338

  TARGET ITEMS (Push Attack Goal)
  --------------------------------
  MovieID 397     : 4.000 → 4.488  (change: +0.488, 298 fake ratings)
  MovieID 1504    : 4.000 → 4.488  (change: +0.488, 298 fake ratings)
  MovieID 3116    : 3.000 → 4.485  (change: +1.485, 298 fake ratings)
  MovieID 2977    : 5.000 → 4.492  (change: -0.508, 298 fake ratings)
  MovieID 2063    : 1.000 → 4.478  (change: +3.478, 298 fake ratings)

  OUTPUT FILES
  ------------
  injection_output/ml100k_original.csv    → Clean 100K ground tru

# UNRR Framework -- Full Evaluation

This section implements the complete UNRR evaluation pipeline:

```
UNRR Framework -- evaluation ready
  Pre-filter . Noise reweighting . Uncertainty estimation
  Baseline: Collaborative Filtering . Matrix Factorisation
        |
  RMSE   MAE   Precision@K   Recall@K
        |
  Dependable recommendations
  Robust against natural and malicious noise
```

**Steps covered:**
- Step A: Rebuild UNRR pipeline (Pre-filter -> Reweighting -> Uncertainty)
- Step B: Baseline models (User-User CF + SVD Matrix Factorisation)
- Step C: RMSE, MAE, Precision@K, Recall@K evaluation
- Step D: Noise level stress test (2% to 20% contamination)
- Step E: Per-attack-type evaluation
- Step F: Ablation study (component contribution)
- Step G: Full visualisations
- Step H: Dependable recommendations output


## Additional Imports for UNRR Evaluation

In [33]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import time
print('UNRR evaluation imports loaded')

UNRR evaluation imports loaded


## Step A -- Load Injected Dataset & Rebuild UNRR Pipeline

Loads `ml100k_injected.csv` from the injection phase and rebuilds all three UNRR stages:
- **Stage 1:** Pre-filter (remove high-suspicion users, score >= 2)
- **Stage 2:** Noise reweighting (trust weight = 1 - SuspicionScore/4 * 0.8)
- **Stage 3:** Uncertainty estimation (item confidence = 1/(1+FakeRatio))


In [34]:
print("=" * 60)
print("  STEP A -- LOAD INJECTED DATASET & UNRR PIPELINE")
print("=" * 60)

INJECT_OUTPUT_DIR = "injection_output"
EVAL_OUTPUT_DIR   = "eval_figures"
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

def save_eval(name):
    path = os.path.join(EVAL_OUTPUT_DIR, name)
    plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.close()
    print(f"  [Saved] {path}")

injected_df  = pd.read_csv(os.path.join(INJECT_OUTPUT_DIR, "ml100k_injected.csv"))
original_df  = pd.read_csv(os.path.join(INJECT_OUTPUT_DIR, "ml100k_original.csv"))
fake_profiles = pd.read_csv(os.path.join(INJECT_OUTPUT_DIR, "malicious_profiles.csv"))

injected_df["AttackType"] = injected_df["AttackType"].fillna("None")
genuine_df   = injected_df[injected_df["IsMalicious"] == False].copy()
malicious_df = injected_df[injected_df["IsMalicious"] == True].copy()

gm_inj = injected_df["Rating"].mean()
item_pop_inj = injected_df.groupby("MovieID")["Rating"].count()
popular_movies_inj = item_pop_inj[item_pop_inj > item_pop_inj.quantile(0.80)].index
print(f"  Injected dataset  : {len(injected_df):,} ratings")
print(f"  Genuine ratings   : {len(genuine_df):,}")
print(f"  Malicious ratings : {len(malicious_df):,}")
print(f"  Global mean       : {gm_inj:.4f}")
print(f"  Contamination     : {len(malicious_df)/len(injected_df)*100:.2f}%")
print(f"  Columns verified  : {injected_df.columns.tolist()}")

  STEP A -- LOAD INJECTED DATASET & UNRR PIPELINE
  Injected dataset  : 116,390 ratings
  Genuine ratings   : 100,000
  Malicious ratings : 16,390
  Global mean       : 3.5689
  Contamination     : 14.08%
  Columns verified  : ['UserID', 'MovieID', 'Rating', 'Timestamp', 'Gender', 'AgeGroup', 'OccupationName', 'Title', 'Genres', 'AttackType', 'IsFiller', 'IsTarget', 'IsMalicious']


In [35]:
# A.2 Per-user features and suspicion score
print("\n  A.2 Computing user features and suspicion scores...")

unrr_user_stats = injected_df.groupby("UserID").agg(
    AvgRating   = ("Rating", "mean"),
    StdRating   = ("Rating", "std"),
    NumRatings  = ("Rating", "count"),
    IsMalicious = ("IsMalicious", "max")
).reset_index()
unrr_user_stats["StdRating"] = unrr_user_stats["StdRating"].fillna(0)

filler_counts_inj = (
    injected_df[injected_df["MovieID"].isin(popular_movies_inj)]
    .groupby("UserID").size().rename("FillerCount"))
unrr_user_stats = unrr_user_stats.merge(filler_counts_inj, on="UserID", how="left")
unrr_user_stats["FillerCount"] = unrr_user_stats["FillerCount"].fillna(0)
unrr_user_stats["FillerRatio"] = unrr_user_stats["FillerCount"] / unrr_user_stats["NumRatings"]
unrr_user_stats["MeanDev"]     = np.abs(unrr_user_stats["AvgRating"] - gm_inj)

# Burst detection from timestamps if available
if "Timestamp" in original_df.columns:
    ts_df = original_df[["UserID","Timestamp"]].copy()
    ts_df["Timestamp"] = pd.to_datetime(ts_df["Timestamp"], errors="coerce")
    ts_df = ts_df.sort_values(["UserID","Timestamp"])
    ts_df["TimeDiff"] = ts_df.groupby("UserID")["Timestamp"].diff().dt.total_seconds().fillna(9999)
    burst_inj = ts_df[ts_df["TimeDiff"] < 3600].groupby("UserID").size().rename("BurstCount")
    unrr_user_stats = unrr_user_stats.merge(burst_inj, on="UserID", how="left")
    unrr_user_stats["BurstCount"] = unrr_user_stats["BurstCount"].fillna(0)
else:
    unrr_user_stats["BurstCount"] = 0

# Composite suspicion score (0-4)
unrr_user_stats["S_Filler"]  = (unrr_user_stats["FillerRatio"] > 0.80).astype(int)
unrr_user_stats["S_LowVar"]  = (unrr_user_stats["StdRating"]   < 0.30).astype(int)
unrr_user_stats["S_HighDev"] = (unrr_user_stats["MeanDev"]      > 1.50).astype(int)
unrr_user_stats["S_Burst"]   = (unrr_user_stats["BurstCount"]   > 10  ).astype(int)
unrr_user_stats["SuspicionScore"] = (
    unrr_user_stats[["S_Filler","S_LowVar","S_HighDev","S_Burst"]].sum(axis=1))

print(f"  Users profiled: {len(unrr_user_stats):,}")
print("  Score distribution:")
print(unrr_user_stats["SuspicionScore"].value_counts().sort_index().to_string())


  A.2 Computing user features and suspicion scores...
  Users profiled: 6,268
  Score distribution:
SuspicionScore
0    2051
1    3526
2     668
3      23


In [36]:
# A.3 Stage 1 -- Pre-filter
print("\n  Stage 1 -- Pre-filter (Remove High-Risk Users)")
high_risk_uids = set(unrr_user_stats[unrr_user_stats["SuspicionScore"] >= 3]["UserID"])
filtered_df    = injected_df[~injected_df["UserID"].isin(high_risk_uids)].copy()
print(f"    High-risk users removed   : {len(high_risk_uids):,}")
print(f"    Ratings before filter     : {len(injected_df):,}")
print(f"    Ratings after filter      : {len(filtered_df):,}")
print(f"    Remaining contamination   : {filtered_df["IsMalicious"].sum()/len(filtered_df)*100:.2f}%")

# A.4 Stage 2 -- Trust weight reweighting
print("\n  Stage 2 -- Noise Reweighting (Trust Weights)")
trust_weights_df = unrr_user_stats[["UserID","SuspicionScore"]].copy()
trust_weights_df["TrustWeight"] = (
    1.0 - (trust_weights_df["SuspicionScore"] / 4.0) * 0.8).clip(0.10, 1.00)
filtered_df = filtered_df.merge(trust_weights_df[["UserID","TrustWeight"]], on="UserID", how="left")
filtered_df["TrustWeight"]    = filtered_df["TrustWeight"].fillna(1.0)
filtered_df["WeightedRating"] = filtered_df["Rating"] * filtered_df["TrustWeight"]

gen_tw  = trust_weights_df[trust_weights_df["UserID"].isin(genuine_df["UserID"].unique())]["TrustWeight"].mean()
mal_tw  = trust_weights_df[trust_weights_df["UserID"].isin(malicious_df["UserID"].unique())]["TrustWeight"].mean()
print(f"    Mean trust weight (genuine)   : {gen_tw:.4f}")
print(f"    Mean trust weight (malicious) : {mal_tw:.4f}")

# A.5 Stage 3 -- Item uncertainty / confidence scoring
print("\n  Stage 3 -- Uncertainty Estimation (Item Confidence)")
item_stats_inj = injected_df.groupby("MovieID").agg(
    TotalRatings = ("Rating","count"),
    FakeRatings  = ("IsMalicious","sum")
).reset_index()
item_stats_inj["FakeRatio"]  = item_stats_inj["FakeRatings"] / item_stats_inj["TotalRatings"]
item_stats_inj["Confidence"] = 1.0 / (1.0 + item_stats_inj["FakeRatio"].fillna(0))
filtered_df = filtered_df.merge(item_stats_inj[["MovieID","Confidence"]], on="MovieID", how="left")
filtered_df["Confidence"]     = filtered_df["Confidence"].fillna(1.0)
filtered_df["AdjustedRating"] = (
    filtered_df["WeightedRating"] * filtered_df["Confidence"] +
    filtered_df["Rating"] * (1 - filtered_df["Confidence"]))

clean_conf = item_stats_inj[item_stats_inj["FakeRatings"]==0]["Confidence"].mean()
atk_conf   = item_stats_inj[item_stats_inj["FakeRatings"]>0]["Confidence"].mean()
print(f"    Mean item confidence (clean)   : {clean_conf:.4f}")
print(f"    Mean item confidence (attacked): {atk_conf:.4f}")
print("\n  UNRR Pipeline rebuilt successfully")


  Stage 1 -- Pre-filter (Remove High-Risk Users)
    High-risk users removed   : 23
    Ratings before filter     : 116,390
    Ratings after filter      : 116,365
    Remaining contamination   : 14.08%

  Stage 2 -- Noise Reweighting (Trust Weights)
    Mean trust weight (genuine)   : 0.8411
    Mean trust weight (malicious) : 0.8738

  Stage 3 -- Uncertainty Estimation (Item Confidence)
    Mean item confidence (clean)   : 1.0000
    Mean item confidence (attacked): 0.8164

  UNRR Pipeline rebuilt successfully


## Step B -- Baseline Models

- **User-User CF:** Memory-based, vectorised cosine similarity, mean-centred prediction
- **SVD Matrix Factorisation:** k=50 latent factors via scipy svds

In [37]:
# Step B -- Model helper functions

def split_ratings(df, rating_col="Rating"):
    data = df[["UserID","MovieID", rating_col]].copy()
    data = data.rename(columns={rating_col: "Rating"})
    data = data.drop_duplicates(subset=["UserID","MovieID"])
    return train_test_split(data, test_size=0.2, random_state=42)

def build_cf(train, n_users=300, n_items=200, n_neighbors=20):
    """Fast vectorised User-User CF using precomputed cosine similarity matrix."""
    gm = train["Rating"].mean()
    um = train.groupby("UserID")["Rating"].mean().to_dict()
    au = train.groupby("UserID").size().nlargest(n_users).index
    ai = train.groupby("MovieID").size().nlargest(n_items).index
    sub   = train[train["UserID"].isin(au) & train["MovieID"].isin(ai)]
    pivot = sub.pivot_table(index="UserID", columns="MovieID", values="Rating")
    pivot = pivot.fillna(pivot.mean())
    mat   = pivot.values.astype(np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-8
    sim   = (mat / norms) @ (mat / norms).T
    uid_idx = {u: i for i, u in enumerate(pivot.index)}
    mid_idx = {m: i for i, m in enumerate(pivot.columns)}
    return {"sim": sim, "pivot": pivot, "uid_idx": uid_idx,
            "mid_idx": mid_idx, "um": um, "gm": gm, "n": n_neighbors}

def predict_cf(test, model):
    """Mean-centred weighted prediction using precomputed similarity matrix."""
    sim, pivot = model["sim"], model["pivot"]
    uid_idx, mid_idx = model["uid_idx"], model["mid_idx"]
    um, gm, n = model["um"], model["gm"], model["n"]
    preds = []
    for _, row in test.iterrows():
        uid, mid = row["UserID"], row["MovieID"]
        u_mean = um.get(uid, gm)
        if uid not in uid_idx or mid not in mid_idx:
            preds.append(u_mean); continue
        ui = uid_idx[uid]; mi = mid_idx[mid]
        s  = sim[ui].copy(); s[ui] = -1
        tk = np.argpartition(s, -n)[-n:]
        ts = s[tk]; rk = pivot.iloc[tk, mi].values
        nm = np.array([um.get(pivot.index[j], gm) for j in tk])
        pred = u_mean + np.dot(ts, rk - nm) / (np.abs(ts).sum() + 1e-8)
        preds.append(float(np.clip(pred, 1, 5)))
    return np.array(preds)

def build_svd(train, k=50):
    """Sparse SVD-based matrix factorisation with k latent factors."""
    gm  = train["Rating"].mean()
    u2i = {u: i for i, u in enumerate(train["UserID"].unique())}
    m2i = {m: i for i, m in enumerate(train["MovieID"].unique())}
    rows = [u2i[u] for u in train["UserID"]]
    cols = [m2i[m] for m in train["MovieID"]]
    vals = list(train["Rating"] - gm)
    mat  = csr_matrix((vals, (rows, cols)), shape=(len(u2i), len(m2i)))
    kk   = min(k, min(len(u2i), len(m2i)) - 1)
    U, s, Vt   = svds(mat.astype(float), k=kk)
    pred_mat   = (U @ np.diag(s) @ Vt) + gm
    return {"pred_mat": pred_mat, "u2i": u2i, "m2i": m2i, "gm": gm}

def predict_svd(test, model):
    """Predict ratings using precomputed SVD decomposition."""
    pm, u2i, m2i, gm = model["pred_mat"], model["u2i"], model["m2i"], model["gm"]
    preds = []
    for _, row in test.iterrows():
        uid, mid = row["UserID"], row["MovieID"]
        p = pm[u2i[uid], m2i[mid]] if (uid in u2i and mid in m2i) else gm
        preds.append(float(np.clip(p, 1, 5)))
    return np.array(preds)

print("Model helper functions defined")

Model helper functions defined


## Step C -- Evaluation Metrics: RMSE, MAE, Precision@K, Recall@K, NDCG@K

- **RMSE** -- Root Mean Square Error (prediction accuracy, penalises large errors)
- **MAE** -- Mean Absolute Error (all errors equal weight)
- **Precision@K** -- Fraction of top-K recommendations that are relevant
- **Recall@K** -- Fraction of all relevant items appearing in top-K
- **NDCG@K** -- Normalised Discounted Cumulative Gain (ranking quality)


In [38]:
# Step C -- Metric functions and evaluation runner

def compute_precision_recall_k(test, preds, k=10, threshold=4.0):
    t = test.copy(); t["pred"] = preds
    p_list, r_list = [], []
    for uid, g in t.groupby("UserID"):
        relevant = set(g[g["Rating"] >= threshold]["MovieID"])
        if len(relevant) == 0: continue
        top_k = set(g.nlargest(k, "pred")["MovieID"])
        hits  = len(top_k & relevant)
        p_list.append(hits / k)
        r_list.append(hits / len(relevant))
    return np.mean(p_list), np.mean(r_list)

def compute_ndcg_k(test, preds, k=10, threshold=4.0):
    t = test.copy(); t["pred"] = preds
    ndcg_list = []
    for uid, g in t.groupby("UserID"):
        relevant = set(g[g["Rating"] >= threshold]["MovieID"])
        if len(relevant) == 0: continue
        top_k = g.nlargest(k, "pred")["MovieID"].tolist()
        dcg   = sum(1/np.log2(i+2) for i, m in enumerate(top_k) if m in relevant)
        idcg  = sum(1/np.log2(i+2) for i in range(min(len(relevant), k)))
        ndcg_list.append(dcg/idcg if idcg > 0 else 0)
    return np.mean(ndcg_list)

def run_full_evaluation(df, label, rating_col="Rating"):
    train, test = split_ratings(df, rating_col=rating_col)
    tv = test["Rating"].values
    results = {}
    for mname, bfn, pfn in [("CF_UU", build_cf, predict_cf),
                              ("SVD_MF", build_svd, predict_svd)]:
        t0    = time.time()
        model = bfn(train)
        preds = pfn(test.reset_index(drop=True), model)
        t_end = time.time() - t0
        rmse  = np.sqrt(mean_squared_error(tv, preds))
        mae   = mean_absolute_error(tv, preds)
        p5,  r5  = compute_precision_recall_k(test.reset_index(drop=True), preds, k=5)
        p10, r10 = compute_precision_recall_k(test.reset_index(drop=True), preds, k=10)
        nd10 = compute_ndcg_k(test.reset_index(drop=True), preds, k=10)
        results[mname] = {"RMSE": rmse, "MAE": mae, "P@5": p5, "R@5": r5,
                           "P@10": p10, "R@10": r10, "NDCG@10": nd10,
                           "Time": t_end, "preds": preds, "test": test}
        print(f"  {label:<22} {mname:<8}  "
              f"RMSE={rmse:.4f}  MAE={mae:.4f}  "
              f"P@10={p10:.4f}  R@10={r10:.4f}  NDCG@10={nd10:.4f}  ({t_end:.1f}s)")
    return results

print("\n[C] RUNNING MODELS ON THREE DATASET VARIANTS")
print("-" * 60)

res_injected = run_full_evaluation(injected_df,  "Injected (Baseline)")
res_filtered = run_full_evaluation(filtered_df,  "UNRR Filtered",   rating_col="AdjustedRating")
res_genuine  = run_full_evaluation(genuine_df,   "Genuine (clean)")



[C] RUNNING MODELS ON THREE DATASET VARIANTS
------------------------------------------------------------
  Injected (Baseline)    CF_UU     RMSE=1.0570  MAE=0.8257  P@10=0.2761  R@10=0.9666  NDCG@10=0.8644  (2.2s)
  Injected (Baseline)    SVD_MF    RMSE=1.1521  MAE=0.9580  P@10=0.2777  R@10=0.9689  NDCG@10=0.8824  (3.6s)
  UNRR Filtered          CF_UU     RMSE=0.9027  MAE=0.7017  P@10=0.1926  R@10=0.9626  NDCG@10=0.7721  (2.3s)
  UNRR Filtered          SVD_MF    RMSE=1.0175  MAE=0.8223  P@10=0.1942  R@10=0.9663  NDCG@10=0.8057  (3.4s)
  Genuine (clean)        CF_UU     RMSE=1.0723  MAE=0.8519  P@10=0.2629  R@10=0.9720  NDCG@10=0.8711  (2.0s)
  Genuine (clean)        SVD_MF    RMSE=1.1146  MAE=0.9292  P@10=0.2638  R@10=0.9728  NDCG@10=0.8825  (2.9s)


In [39]:
# C.2 Consolidated results table
print("\n[C.2] CONSOLIDATED RESULTS TABLE")

eval_rows = []
for ds_label, res in [("Injected", res_injected),
                       ("UNRR Filtered", res_filtered),
                       ("Genuine", res_genuine)]:
    for model, m in res.items():
        eval_rows.append({"Dataset": ds_label, "Model": model,
                           "RMSE": round(m["RMSE"],4), "MAE": round(m["MAE"],4),
                           "P@5":  round(m["P@5"],4),  "R@5": round(m["R@5"],4),
                           "P@10": round(m["P@10"],4), "R@10": round(m["R@10"],4),
                           "NDCG@10": round(m["NDCG@10"],4)})

eval_table = pd.DataFrame(eval_rows)
print(eval_table.to_string(index=False))

print("\n  Improvement: UNRR Filtered vs Injected (Baseline)")
print(f"  {chr(34)}Model{chr(34):<10} {chr(34)}dRMSE{chr(34):>8} {chr(34)}dMAE{chr(34):>8} {chr(34)}dP@10{chr(34):>8} {chr(34)}dR@10{chr(34):>8} {chr(34)}dNDCG@10{chr(34):>10}")
for model in ["CF_UU","SVD_MF"]:
    ri = eval_table[(eval_table["Dataset"]=="Injected")      & (eval_table["Model"]==model)].iloc[0]
    rf = eval_table[(eval_table["Dataset"]=="UNRR Filtered") & (eval_table["Model"]==model)].iloc[0]
    print(f"  {model:<10} "
          f"{rf["RMSE"]-ri["RMSE"]:>+8.4f} "
          f"{rf["MAE"] -ri["MAE"]:>+8.4f} "
          f"{rf["P@10"]-ri["P@10"]:>+8.4f} "
          f"{rf["R@10"]-ri["R@10"]:>+8.4f} "
          f"{rf["NDCG@10"]-ri["NDCG@10"]:>+10.4f}")


[C.2] CONSOLIDATED RESULTS TABLE
      Dataset  Model   RMSE    MAE    P@5    R@5   P@10   R@10  NDCG@10
     Injected  CF_UU 1.0570 0.8257 0.4274 0.8619 0.2761 0.9666   0.8644
     Injected SVD_MF 1.1521 0.9580 0.4357 0.8725 0.2777 0.9689   0.8824
UNRR Filtered  CF_UU 0.9027 0.7017 0.3012 0.8374 0.1926 0.9626   0.7721
UNRR Filtered SVD_MF 1.0175 0.8223 0.3138 0.8571 0.1942 0.9663   0.8057
      Genuine  CF_UU 1.0723 0.8519 0.4213 0.8783 0.2629 0.9720   0.8711
      Genuine SVD_MF 1.1146 0.9292 0.4272 0.8879 0.2638 0.9728   0.8825

  Improvement: UNRR Filtered vs Injected (Baseline)
  "Model"          "dRMSE       " "dMAE       " "dP@10       " "dR@10       " "dNDCG@10         "
  CF_UU       -0.1543  -0.1240  -0.0835  -0.0040    -0.0923
  SVD_MF      -0.1346  -0.1357  -0.0835  -0.0026    -0.0767


## Step D -- Noise Level Stress Test

Tests RMSE and MAE at contamination levels 2%, 5%, 10%, 15%, 20%.
Shows how UNRR maintains accuracy as attack intensity increases.


In [40]:
print("\n[D] NOISE LEVEL STRESS TEST")
print("-" * 50)
print("  Testing contamination levels: 2%, 5%, 10%, 15%, 20%")

noise_levels = [0.02, 0.05, 0.10, 0.15, 0.20]
stress_rows  = []

for noise_pct in noise_levels:
    n_fake = int(len(genuine_df) * noise_pct / (1 - noise_pct))
    fake_s = fake_profiles.sample(n=min(n_fake, len(fake_profiles)),
                                   replace=True, random_state=42)
    fake_s = fake_s[["UserID","MovieID","Rating","IsMalicious","AttackType"]].copy()
    s_data = pd.concat([
        genuine_df[["UserID","MovieID","Rating","IsMalicious","AttackType"]],
        fake_s], ignore_index=True).drop_duplicates(subset=["UserID","MovieID"])

    gm_s  = s_data["Rating"].mean()
    ip_s  = s_data.groupby("MovieID")["Rating"].count()
    pop_s = ip_s[ip_s > ip_s.quantile(0.80)].index
    us_s  = s_data.groupby("UserID").agg(
        AvgRating=("Rating","mean"), StdRating=("Rating","std"),
        NumRatings=("Rating","count")).reset_index()
    us_s["StdRating"] = us_s["StdRating"].fillna(0)
    fc_s  = s_data[s_data["MovieID"].isin(pop_s)].groupby("UserID").size()
    us_s  = us_s.merge(fc_s.rename("FC"), on="UserID", how="left")
    us_s["FC"]  = us_s["FC"].fillna(0)
    us_s["FR"]  = us_s["FC"] / us_s["NumRatings"]
    us_s["MD"]  = np.abs(us_s["AvgRating"] - gm_s)
    us_s["SS"]  = ((us_s["FR"] > 0.80).astype(int) +
                   (us_s["StdRating"] < 0.30).astype(int) +
                   (us_s["MD"] > 1.50).astype(int))
    us_s["TW"]  = (1.0 - (us_s["SS"] / 4.0) * 0.8).clip(0.10, 1.00)
    hr_s  = set(us_s[us_s["SS"] >= 2]["UserID"])
    flt_s = s_data[~s_data["UserID"].isin(hr_s)].copy()
    flt_s = flt_s.merge(us_s[["UserID","TW"]], on="UserID", how="left")
    flt_s["TW"]   = flt_s["TW"].fillna(1.0)
    flt_s["AdjR"] = flt_s["Rating"] * flt_s["TW"]

    for ds_lbl, d_use, rcol in [("Baseline", s_data, "Rating"), ("UNRR", flt_s, "AdjR")]:
        try:
            tr, te = split_ratings(d_use, rating_col=rcol)
            sv_m   = build_svd(tr)
            preds  = predict_svd(te.reset_index(drop=True), sv_m)
            rmse   = np.sqrt(mean_squared_error(te["Rating"].values, preds))
            mae    = mean_absolute_error(te["Rating"].values, preds)
            stress_rows.append({"NoiseLevel": noise_pct*100,
                                 "Dataset": ds_lbl, "RMSE": rmse, "MAE": mae})
        except Exception as e:
            print(f"    Warning {noise_pct} {ds_lbl}: {e}")
    print(f"  Noise {noise_pct*100:4.0f}% done")

stress_results = pd.DataFrame(stress_rows)
print("\n  Stress Test Results (SVD_MF):")
print(stress_results.pivot_table(index="NoiseLevel", columns="Dataset",
                                  values=["RMSE","MAE"]).round(4).to_string())


[D] NOISE LEVEL STRESS TEST
--------------------------------------------------
  Testing contamination levels: 2%, 5%, 10%, 15%, 20%
  Noise    2% done
  Noise    5% done
  Noise   10% done
  Noise   15% done
  Noise   20% done

  Stress Test Results (SVD_MF):
                MAE             RMSE        
Dataset    Baseline    UNRR Baseline    UNRR
NoiseLevel                                  
2.0          0.9288  0.9149   1.1125  1.0973
5.0          0.9407  0.9123   1.1262  1.0942
10.0         0.9447  0.9057   1.1335  1.0891
15.0         0.9557  0.9029   1.1467  1.0868
20.0         0.9557  0.9029   1.1467  1.0868


## Step E -- Per-Attack-Type Evaluation

Evaluates UNRR separately for each attack type: Random, Average, Bandwagon, LoveHate.


In [41]:
print("\n[E] PER-ATTACK-TYPE EVALUATION")
print("-" * 50)

attack_types_eval = [a for a in injected_df["AttackType"].unique() if a != "None"]
atk_rows = []

for atk in attack_types_eval:
    atk_fake = fake_profiles[fake_profiles["AttackType"]==atk][
                   ["UserID","MovieID","Rating","IsMalicious","AttackType"]]
    atk_data = pd.concat([
        genuine_df[["UserID","MovieID","Rating","IsMalicious","AttackType"]],
        atk_fake], ignore_index=True).drop_duplicates(subset=["UserID","MovieID"])

    gm_a  = atk_data["Rating"].mean()
    pop_a = atk_data.groupby("MovieID")["Rating"].count()
    pop_a = pop_a[pop_a > pop_a.quantile(0.80)].index
    us_a  = atk_data.groupby("UserID").agg(
        AvgRating=("Rating","mean"), StdRating=("Rating","std"),
        NumRatings=("Rating","count")).reset_index()
    us_a["StdRating"] = us_a["StdRating"].fillna(0)
    fc_a  = atk_data[atk_data["MovieID"].isin(pop_a)].groupby("UserID").size()
    us_a  = us_a.merge(fc_a.rename("FC"), on="UserID", how="left")
    us_a["FC"]  = us_a["FC"].fillna(0)
    us_a["FR"]  = us_a["FC"] / us_a["NumRatings"]
    us_a["MD"]  = np.abs(us_a["AvgRating"] - gm_a)
    us_a["SS"]  = ((us_a["FR"] > 0.80).astype(int) +
                   (us_a["StdRating"] < 0.30).astype(int) +
                   (us_a["MD"] > 1.50).astype(int))
    us_a["TW"]  = (1.0 - (us_a["SS"] / 4.0) * 0.8).clip(0.10, 1.00)
    hr_a  = set(us_a[us_a["SS"] >= 2]["UserID"])
    flt_a = atk_data[~atk_data["UserID"].isin(hr_a)].copy()
    flt_a = flt_a.merge(us_a[["UserID","TW"]], on="UserID", how="left")
    flt_a["TW"]   = flt_a["TW"].fillna(1.0)
    flt_a["AdjR"] = flt_a["Rating"] * flt_a["TW"]

    for ds_lbl, d_use, rcol in [("Baseline", atk_data, "Rating"), ("UNRR", flt_a, "AdjR")]:
        try:
            tr, te = split_ratings(d_use, rating_col=rcol)
            sv_m   = build_svd(tr)
            preds  = predict_svd(te.reset_index(drop=True), sv_m)
            rmse   = np.sqrt(mean_squared_error(te["Rating"].values, preds))
            mae    = mean_absolute_error(te["Rating"].values, preds)
            p10,r10 = compute_precision_recall_k(te.reset_index(drop=True), preds, k=10)
            atk_rows.append({"AttackType": atk, "Dataset": ds_lbl,
                              "RMSE": rmse, "MAE": mae, "P@10": p10, "R@10": r10})
        except Exception as e:
            print(f"    Warning {atk} {ds_lbl}: {e}")
    print(f"  {atk} attack evaluated")

attack_results = pd.DataFrame(atk_rows)
print("\n  Per-Attack-Type Results:")
print(attack_results.round(4).to_string(index=False))


[E] PER-ATTACK-TYPE EVALUATION
--------------------------------------------------
  Random attack evaluated
  Average attack evaluated
  Bandwagon attack evaluated
  Love-Push attack evaluated
  Hate-Nuke attack evaluated

  Per-Attack-Type Results:
AttackType  Dataset   RMSE    MAE   P@10   R@10
    Random Baseline 1.1297 0.9441 0.2658 0.9706
    Random     UNRR 1.1070 0.9249 0.2690 0.9670
   Average Baseline 1.1069 0.9217 0.2641 0.9717
   Average     UNRR 1.0870 0.9041 0.2718 0.9683
 Bandwagon Baseline 1.1087 0.9206 0.2712 0.9700
 Bandwagon     UNRR 1.0853 0.9046 0.2724 0.9675
 Love-Push Baseline 1.1133 0.9307 0.2686 0.9723
 Love-Push     UNRR 1.0900 0.9078 0.2750 0.9659
 Hate-Nuke Baseline 1.1548 0.9669 0.2625 0.9736
 Hate-Nuke     UNRR 1.0966 0.9145 0.2656 0.9688


## Step F -- Ablation Study: UNRR Component Contribution

Four configurations tested:
1. **No UNRR** -- Raw injected data
2. **Stage 1 Only** -- Pre-filter only
3. **Stage 1+2** -- Filter + reweighting
4. **Full UNRR** -- All three stages


In [42]:
print("\n[F] ABLATION STUDY")
print("-" * 55)

tr_v1, te_v1 = split_ratings(injected_df)
sv_v1 = build_svd(tr_v1); p_v1 = predict_svd(te_v1.reset_index(drop=True), sv_v1)

flt_v2 = injected_df[~injected_df["UserID"].isin(high_risk_uids)].copy()
tr_v2, te_v2 = split_ratings(flt_v2)
sv_v2 = build_svd(tr_v2); p_v2 = predict_svd(te_v2.reset_index(drop=True), sv_v2)

flt_v3 = filtered_df.copy(); flt_v3["S12R"] = flt_v3["WeightedRating"]
tr_v3, te_v3 = split_ratings(flt_v3, rating_col="S12R")
sv_v3 = build_svd(tr_v3); p_v3 = predict_svd(te_v3.reset_index(drop=True), sv_v3)

tr_v4, te_v4 = split_ratings(filtered_df, rating_col="AdjustedRating")
sv_v4 = build_svd(tr_v4); p_v4 = predict_svd(te_v4.reset_index(drop=True), sv_v4)

ablation_rows = []
for label, te, preds in [
    ("No UNRR (Baseline)",     te_v1, p_v1),
    ("Stage 1 Only (Filter)",  te_v2, p_v2),
    ("Stage 1+2 (Reweight)",   te_v3, p_v3),
    ("Full UNRR (All Stages)", te_v4, p_v4)]:
    tv   = te["Rating"].values
    rmse = np.sqrt(mean_squared_error(tv, preds))
    mae  = mean_absolute_error(tv, preds)
    p10, r10 = compute_precision_recall_k(te.reset_index(drop=True), preds, k=10)
    nd10 = compute_ndcg_k(te.reset_index(drop=True), preds, k=10)
    ablation_rows.append({"Configuration": label, "RMSE": rmse, "MAE": mae,
                           "P@10": p10, "R@10": r10, "NDCG@10": nd10})
    print(f"  {label:<30}  RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"P@10={p10:.4f}  NDCG@10={nd10:.4f}")

ablation_df = pd.DataFrame(ablation_rows)


[F] ABLATION STUDY
-------------------------------------------------------
  No UNRR (Baseline)              RMSE=1.1521  MAE=0.9580  P@10=0.2777  NDCG@10=0.8824
  Stage 1 Only (Filter)           RMSE=1.1451  MAE=0.9525  P@10=0.2834  NDCG@10=0.8849
  Stage 1+2 (Reweight)            RMSE=1.0081  MAE=0.8135  P@10=0.1943  NDCG@10=0.8089
  Full UNRR (All Stages)          RMSE=1.0175  MAE=0.8223  P@10=0.1942  NDCG@10=0.8057


In [43]:
"""
ABLATION STUDY - UNRR Component Contribution
Includes: RMSE, MAE, Precision@5, Recall@5, NDCG@5
Generates visualization matching the reference image
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error

# HELPER FUNCTIONS FOR EVALUATION

def compute_precision_recall_k(test_df, predictions, k=5):
    """
    Compute Precision@K and Recall@K
    
    Args:
        test_df: DataFrame with 'UserID', 'ProductID', 'Rating'
        predictions: Predicted ratings array
        k: Top-K to evaluate
    
    Returns:
        precision_k, recall_k
    """
    test_df = test_df.copy()
    test_df['Prediction'] = predictions
    
    # Get top-k predictions per user
    top_k_pred = test_df.groupby('UserID').apply(
        lambda x: x.nlargest(k, 'Prediction')['ProductID'].values
    )
    
    # Get actual high ratings (>= 4) per user
    actual_high = test_df.groupby('UserID').apply(
        lambda x: x[x['Rating'] >= 4]['ProductID'].values
    )
    
    precisions = []
    recalls = []
    
    for uid in test_df['UserID'].unique():
        if uid in top_k_pred.index and uid in actual_high.index:
            pred_set = set(top_k_pred[uid])
            actual_set = set(actual_high[uid])
            
            if len(pred_set) > 0:
                precision = len(pred_set & actual_set) / len(pred_set)
                precisions.append(precision)
            
            if len(actual_set) > 0:
                recall = len(pred_set & actual_set) / len(actual_set)
                recalls.append(recall)
    
    return np.mean(precisions) if precisions else 0, np.mean(recalls) if recalls else 0


def compute_ndcg_k(test_df, predictions, k=5):
    """
    Compute NDCG@K (Normalized Discounted Cumulative Gain)
    
    Args:
        test_df: DataFrame with 'UserID', 'ProductID', 'Rating'
        predictions: Predicted ratings array
        k: Top-K to evaluate
    
    Returns:
        ndcg_k (average across users)
    """
    test_df = test_df.copy()
    test_df['Prediction'] = predictions
    
    ndcg_scores = []
    
    for uid in test_df['UserID'].unique():
        user_data = test_df[test_df['UserID'] == uid]
        
        # Sort by prediction and take top-k
        top_k_idx = user_data.nlargest(k, 'Prediction').index
        top_k_ratings = user_data.loc[top_k_idx, 'Rating'].values
        
        # Ideal ranking: sorted by actual rating, take top-k
        ideal_ratings = sorted(user_data['Rating'].values, reverse=True)[:k]
        
        # Compute DCG
        dcg = 0
        for i, rating in enumerate(top_k_ratings, 1):
            dcg += (rating - 1) / np.log2(i + 1)  # Rating normalized to relevance
        
        # Compute IDCG
        idcg = 0
        for i, rating in enumerate(ideal_ratings, 1):
            idcg += (rating - 1) / np.log2(i + 1)
        
        # NDCG
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcg_scores.append(ndcg)
    
    return np.mean(ndcg_scores) if ndcg_scores else 0


def split_ratings(df, rating_col='Rating', test_size=0.2):
    """Split ratings into train/test"""
    np.random.seed(42)
    mask = np.random.rand(len(df)) < (1 - test_size)
    return df[mask].copy(), df[~mask].copy()


def build_svd(train_df, n_components=50):
    """Build SVD model"""
    from sklearn.decomposition import TruncatedSVD
    
    users = train_df['UserID'].unique()
    products = train_df['ProductID'].unique()
    
    user_idx = {u: i for i, u in enumerate(users)}
    item_idx = {p: i for i, p in enumerate(products)}
    
    R = np.zeros((len(users), len(products)))
    for _, row in train_df.iterrows():
        u_idx = user_idx.get(row['UserID'])
        i_idx = item_idx.get(row['ProductID'])
        if u_idx is not None and i_idx is not None:
            R[u_idx, i_idx] = row['Rating']
    
    svd = TruncatedSVD(n_components=min(n_components, min(R.shape) - 1), random_state=42)
    U = svd.fit_transform(R)
    V = svd.components_.T
    
    return {'U': U, 'V': V, 'user_idx': user_idx, 'item_idx': item_idx, 'users': users, 'products': products}


def predict_svd(test_df, svd_model):
    """Generate predictions from SVD model"""
    U = svd_model['U']
    V = svd_model['V']
    user_idx = svd_model['user_idx']
    item_idx = svd_model['item_idx']
    
    predictions = []
    for _, row in test_df.iterrows():
        u_idx = user_idx.get(row['UserID'])
        i_idx = item_idx.get(row['ProductID'])
        
        if u_idx is not None and i_idx is not None:
            pred = U[u_idx] @ V[i_idx]
        else:
            pred = 3.0  # Default prediction
        
        predictions.append(np.clip(pred, 1, 5))
    
    return np.array(predictions)


# ABLATION STUDY - MAIN CODE

def run_ablation_study(injected_df, filtered_df, high_risk_uids, filtered_df_reweighted):
    """
    Run ablation study with 4 configurations
    
    Args:
        injected_df: Full dataset with attacks
        filtered_df: After Stage 1 pre-filter
        high_risk_uids: User IDs marked as high-risk
        filtered_df_reweighted: After Stage 1+2 (reweighting)
    
    Returns:
        ablation_df: DataFrame with results
        fig: Matplotlib figure with 3-panel visualization
    """
    
    print("\n[ABLATION STUDY] UNRR Component Contribution")
    print("=" * 90)
    print("Evaluating 4 configurations:")
    print("  1. No UNRR (Baseline) - Original contaminated data")
    print("  2. Stage 1 Only (Filter) - Pre-filtering only")
    print("  3. Stage 1+2 (Reweight) - Pre-filter + Reweighting")
    print("  4. Full UNRR (All Stages) - All three stages")
    print("=" * 90)
    
    # Configuration 1: No UNRR (Baseline - all data)
    print("\n[1/4] No UNRR (Baseline)...")
    tr_v1, te_v1 = split_ratings(injected_df)
    sv_v1 = build_svd(tr_v1)
    p_v1 = predict_svd(te_v1.reset_index(drop=True), sv_v1)
    
    # Configuration 2: Stage 1 Only (Pre-filter)
    print("[2/4] Stage 1 Only (Filter)...")
    flt_v2 = injected_df[~injected_df['UserID'].isin(high_risk_uids)].copy()
    tr_v2, te_v2 = split_ratings(flt_v2)
    sv_v2 = build_svd(tr_v2)
    p_v2 = predict_svd(te_v2.reset_index(drop=True), sv_v2)
    
    # Configuration 3: Stage 1+2 (Pre-filter + Reweighting)
    print("[3/4] Stage 1+2 (Reweight)...")
    flt_v3 = filtered_df_reweighted.copy()
    if 'WeightedRating' not in flt_v3.columns:
        flt_v3['WeightedRating'] = flt_v3['Rating']
    tr_v3, te_v3 = split_ratings(flt_v3, rating_col='WeightedRating')
    sv_v3 = build_svd(tr_v3.rename(columns={'WeightedRating': 'Rating'}))
    p_v3 = predict_svd(te_v3.reset_index(drop=True).rename(columns={'WeightedRating': 'Rating'}), sv_v3)
    
    # Configuration 4: Full UNRR (All Stages)
    print("[4/4] Full UNRR (All Stages)...")
    flt_v4 = filtered_df.copy()
    if 'AdjustedRating' not in flt_v4.columns:
        flt_v4['AdjustedRating'] = flt_v4['Rating']
    tr_v4, te_v4 = split_ratings(flt_v4, rating_col='AdjustedRating')
    sv_v4 = build_svd(tr_v4.rename(columns={'AdjustedRating': 'Rating'}))
    p_v4 = predict_svd(te_v4.reset_index(drop=True).rename(columns={'AdjustedRating': 'Rating'}), sv_v4)
    
    # Compute metrics for all configurations
    print("\nComputing metrics...")
    ablation_rows = []
    
    for label, te, preds in [
        ("No UNRR (Baseline)", te_v1, p_v1),
        ("Stage 1 only (Filter)", te_v2, p_v2),
        ("Stage 1+2 (Reweight)", te_v3, p_v3),
        ("Full UNRR (All Stages)", te_v4, p_v4)
    ]:
        # Get rating column name
        rating_col = [c for c in te.columns if 'Rating' in c or 'Adjusted' in c or 'Weighted' in c][0] \
                     if len([c for c in te.columns if 'Rating' in c or 'Adjusted' in c or 'Weighted' in c]) > 0 \
                     else 'Rating'
        
        tv = te[rating_col].values
        
        # Compute metrics
        rmse = np.sqrt(mean_squared_error(tv, preds))
        mae = mean_absolute_error(tv, preds)
        p5, r5 = compute_precision_recall_k(te.reset_index(drop=True), preds, k=5)
        ndcg5 = compute_ndcg_k(te.reset_index(drop=True), preds, k=5)
        
        ablation_rows.append({
            'Configuration': label,
            'RMSE': rmse,
            'MAE': mae,
            'P@5': p5,
            'R@5': r5,
            'NDCG@5': ndcg5
        })
        
        print(f"  {label:<30} RMSE={rmse:.4f}  MAE={mae:.4f}  "
              f"P@5={p5:.4f}  R@5={r5:.4f}  NDCG@5={ndcg5:.4f}")
    
    # Create DataFrame
    ablation_df = pd.DataFrame(ablation_rows)
    
    # Create visualization
    print("\nGenerating visualization...")
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Ablation Study — UNRR Component Contribution\nEach bar shows what is gained by adding the next component',
                 fontsize=14, fontweight='bold', y=1.02)
    
    colors = ['#d32f2f', '#ff9800', '#ffeb3b', '#4caf50']  # Red, Orange, Yellow, Green
    
    # Plot 1: RMSE (Lower is Better)
    axes[0].bar(ablation_df['Configuration'], ablation_df['RMSE'], color=colors, edgecolor='black', linewidth=1.5)
    axes[0].set_title('RMSE by UNRR Configuration\n(Lower = Better)', fontweight='bold', fontsize=12)
    axes[0].set_ylabel('RMSE', fontsize=11)
    axes[0].set_ylim(0, max(ablation_df['RMSE']) * 1.15)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(ablation_df['RMSE']):
        axes[0].text(i, v + 0.05, f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Plot 2: Precision@5 (Higher is Better)
    axes[1].bar(ablation_df['Configuration'], ablation_df['P@5'], color=colors, edgecolor='black', linewidth=1.5)
    axes[1].set_title('Precision@5 by UNRR Configuration\n(Higher = Better)', fontweight='bold', fontsize=12)
    axes[1].set_ylabel('Precision@5', fontsize=11)
    axes[1].set_ylim(0, max(ablation_df['P@5']) * 1.15)
    axes[1].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(ablation_df['P@5']):
        axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Plot 3: NDCG@5 (Higher is Better)
    axes[2].bar(ablation_df['Configuration'], ablation_df['NDCG@5'], color=colors, edgecolor='black', linewidth=1.5)
    axes[2].set_title('NDCG@5 by UNRR Configuration\n(Higher = Better)', fontweight='bold', fontsize=12)
    axes[2].set_ylabel('NDCG@5', fontsize=11)
    axes[2].set_ylim(0, max(ablation_df['NDCG@5']) * 1.15)
    axes[2].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(ablation_df['NDCG@5']):
        axes[2].text(i, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Style all axes
    for ax in axes:
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_axisbelow(True)
    
    plt.tight_layout()
    
    return ablation_df, fig

# EXAMPLE USAGE

if __name__ == "__main__":
    # This code would be called from your main UNRR script like:
    
    # ablation_df, ablation_fig = run_ablation_study(
    #     injected_df=df_work,
    #     filtered_df=df_filtered,
    #     high_risk_uids=high_risk_uids,
    #     filtered_df_reweighted=df_filtered_v2
    # )
    
    # ablation_fig.savefig('ablation_study.png', dpi=120, bbox_inches='tight')
    # print("\n✅ Saved: ablation_study.png")
    # print(ablation_df.to_string(index=False))
    
    print("Ablation study module ready. Call run_ablation_study() from main script.")

Ablation study module ready. Call run_ablation_study() from main script.


## Step G -- Evaluation Visualisations

Eight comprehensive figures covering all evaluation dimensions.


In [44]:
print("\n[G] GENERATING EVALUATION VISUALISATIONS")

DS_ORDER  = ["Injected", "UNRR Filtered", "Genuine"]
DS_COLORS = ["#e05252", "#2a9d8f", "#457b9d"]

# G.1 Main 4-metric comparison
METRICS = [("RMSE","RMSE (lower is better)"), ("MAE","MAE (lower is better)"),
           ("P@10","Precision@10 (higher is better)"), ("R@10","Recall@10 (higher is better)")]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
x = np.arange(len(DS_ORDER)); w = 0.32
for ax, (metric, ylabel) in zip(axes, METRICS):
    for i, model in enumerate(["CF_UU","SVD_MF"]):
        vals = [eval_table[(eval_table["Dataset"]==ds)&(eval_table["Model"]==model)][metric].values[0]
                for ds in DS_ORDER]
        bars = ax.bar(x+(i-0.5)*w, vals, w, label=model, color=DS_COLORS,
                      alpha=1.0 if i==0 else 0.55, hatch="" if i==0 else "///",
                      edgecolor="white", linewidth=0.8)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_title(ylabel, fontsize=9, pad=6); ax.set_ylabel(metric)
    ax.set_xticks(x); ax.set_xticklabels(["Injected\n(Baseline)","UNRR\nFiltered","Genuine\n(clean)"], fontsize=8)
handles = [plt.Rectangle((0,0),1,1,color="grey",alpha=1.0,label="CF_UU"),
           plt.Rectangle((0,0),1,1,color="grey",alpha=0.55,hatch="///",label="SVD_MF")]
axes[0].legend(handles=handles, fontsize=8)
plt.suptitle("Figure G.1 -- UNRR Framework: Full Metric Comparison", fontsize=11, y=1.02)
plt.tight_layout(); save_eval("figG1_main_metrics.png")
print("  Figure G.1 saved")


[G] GENERATING EVALUATION VISUALISATIONS
  [Saved] eval_figures\figG1_main_metrics.png
  Figure G.1 saved


In [45]:
# G.2 Noise stress test
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ["RMSE", "MAE"]):
    for ds, color, ls in [("Baseline","#e05252","-"), ("UNRR","#2a9d8f","--")]:
        sub = stress_results[stress_results["Dataset"]==ds].sort_values("NoiseLevel")
        ax.plot(sub["NoiseLevel"], sub[metric], color=color, linestyle=ls,
                linewidth=2, marker="o", markersize=7, label=ds)
        ax.fill_between(sub["NoiseLevel"], sub[metric], alpha=0.08, color=color)
    ax.set_title(f"Figure G.2 -- {metric} vs Contamination Level (SVD_MF)", fontsize=10)
    ax.set_xlabel("Contamination Rate (%)"); ax.set_ylabel(metric)
    ax.legend(); ax.set_xticks([2,5,10,15,20])
plt.tight_layout(); save_eval("figG2_stress_test.png")
print("  Figure G.2 saved")

# G.3 Per-attack-type
atk_order = attack_results["AttackType"].unique().tolist()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x2 = np.arange(len(atk_order)); w2 = 0.35
for ax, metric in zip(axes, ["RMSE", "P@10"]):
    for i, (ds, color) in enumerate([("Baseline","#e05252"),("UNRR","#2a9d8f")]):
        vals = [attack_results[(attack_results["AttackType"]==a)&
                               (attack_results["Dataset"]==ds)][metric].values[0]
                if len(attack_results[(attack_results["AttackType"]==a)&
                                      (attack_results["Dataset"]==ds)]) else np.nan
                for a in atk_order]
        bars = ax.bar(x2+(i-0.5)*w2, vals, w2, label=ds, color=color,
                      edgecolor="white", alpha=0.85)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                        f"{val:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_title(f"Figure G.3 -- {metric} by Attack Type (SVD_MF)", fontsize=10)
    ax.set_xlabel("Attack Type"); ax.set_ylabel(metric)
    ax.set_xticks(x2); ax.set_xticklabels(atk_order); ax.legend()
plt.tight_layout(); save_eval("figG3_attack_type.png")
print("  Figure G.3 saved")


  [Saved] eval_figures\figG2_stress_test.png
  Figure G.2 saved
  [Saved] eval_figures\figG3_attack_type.png
  Figure G.3 saved


In [59]:
# G.4 Ablation study
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
abl_colors = ["#e05252","#f4a261","#e9c46a","#2a9d8f"]
for ax, metric in zip(axes, ["RMSE","P@10","NDCG@10"]):
    vals = ablation_df[metric].values
    bars = ax.bar(range(4), vals, color=abl_colors, edgecolor="white", linewidth=0.8)
    ax.set_title(f"Figure G.4 -- {metric}: Ablation Study", fontsize=10)
    ax.set_xticks(range(4))
    ax.set_xticklabels(["No UNRR","Stage 1\nOnly","Stage 1+2","Full UNRR"], fontsize=8)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); save_eval("figG4_ablation.png")
print("  Figure G.4 saved")

# G.5 Metric heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["RMSE","NDCG@10"]):
    pivot = eval_table.pivot_table(index="Model", columns="Dataset", values=metric)
    pivot = pivot[[c for c in DS_ORDER if c in pivot.columns]]
    sns.heatmap(pivot, annot=True, fmt=".4f", cmap="RdYlGn_r",
                linewidths=0.5, ax=ax, annot_kws={"size":11})
    ax.set_title(f"Figure G.5 -- {metric} Heatmap", fontsize=10)
plt.tight_layout(); save_eval("figG5_metric_heatmap.png")
print("  Figure G.5 saved")

# G.6 P@10 vs R@10 scatter
fig, ax = plt.subplots(figsize=(8, 6))
markers_map = {"Injected": "o", "UNRR Filtered": "s", "Genuine": "^"}
m_colors   = {"CF_UU": "#e05252", "SVD_MF": "#2a9d8f"}
for _, row in eval_table.iterrows():
    ax.scatter(row["R@10"], row["P@10"], color=m_colors[row["Model"]],
               marker=markers_map.get(row["Dataset"],"o"), s=130, zorder=5)
    ax.annotate(f"{row["Dataset"][:3]}-{row["Model"][:2]}",
                (row["R@10"], row["P@10"]),
                textcoords="offset points", xytext=(5,5), fontsize=8)
ax.set_xlabel("Recall@10"); ax.set_ylabel("Precision@10")
ax.set_title("Figure G.6 -- Precision@10 vs Recall@10", fontsize=11)
plt.tight_layout(); save_eval("figG6_precision_recall.png")
print("  Figure G.6 saved")

# Precision_Recall Barchart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

datasets_order = ["Injected", "UNRR Filtered", "Genuine"]
models         = ["CF_UU", "SVD_MF"]
bar_colors     = {"CF_UU": "#e05252", "SVD_MF": "#2a9d8f"}
x              = np.arange(len(datasets_order))
width          = 0.35

for ax, metric, title in zip(
        axes,
        ["P@10", "R@10"],
        ["Precision@10 — Higher is Better", "Recall@10 — Higher is Better"]):

    for idx, model in enumerate(models):
        vals = []
        for ds in datasets_order:
            row = eval_table[(eval_table["Model"]==model) &
                             (eval_table["Dataset"]==ds)]
            vals.append(float(row[metric].values[0]) if len(row) else 0.0)

        offset = (idx - 0.5) * width
        bars = ax.bar(x + offset, vals, width,
                      label=model, color=bar_colors[model],
                      edgecolor="white", linewidth=0.8, zorder=3)

        # Value labels on top of each bar
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.003,
                    f"{v:.4f}", ha="center", va="bottom",
                    fontsize=8, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(datasets_order, fontsize=10)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)
    ax.legend(title="Model", fontsize=9)

fig.suptitle("Figure G.6 — Precision@10 & Recall@10 by Dataset and Model",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
save_eval("figG6_precision_recall_bar.png")
print("  Figure G.6 bar chart saved")


  [Saved] eval_figures\figG4_ablation.png
  Figure G.4 saved
  [Saved] eval_figures\figG5_metric_heatmap.png
  Figure G.5 saved
  [Saved] eval_figures\figG6_precision_recall.png
  Figure G.6 saved
  [Saved] eval_figures\figG6_precision_recall_bar.png
  Figure G.6 bar chart saved


In [47]:
# G.7 Trust weight and item confidence distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
gen_tw_v = trust_weights_df[trust_weights_df["UserID"].isin(genuine_df["UserID"].unique())]["TrustWeight"]
mal_tw_v = trust_weights_df[trust_weights_df["UserID"].isin(malicious_df["UserID"].unique())]["TrustWeight"]
axes[0].hist(gen_tw_v, bins=20, alpha=0.75, color="#457b9d", edgecolor="white",
             label=f"Genuine  (mean={gen_tw_v.mean():.3f})")
axes[0].hist(mal_tw_v, bins=20, alpha=0.75, color="#e05252", edgecolor="white",
             label=f"Malicious (mean={mal_tw_v.mean():.3f})")
axes[0].set_title("Figure G.7a -- User Trust Weight: Genuine vs Malicious", fontsize=10)
axes[0].set_xlabel("Trust Weight"); axes[0].set_ylabel("Number of Users"); axes[0].legend()

clean_c = item_stats_inj[item_stats_inj["FakeRatings"]==0]["Confidence"]
atk_c   = item_stats_inj[item_stats_inj["FakeRatings"]>0]["Confidence"]
axes[1].hist(clean_c, bins=30, alpha=0.75, color="#2a9d8f", edgecolor="white",
             label=f"Clean items  (mean={clean_c.mean():.3f})")
axes[1].hist(atk_c,   bins=30, alpha=0.75, color="#e05252", edgecolor="white",
             label=f"Attacked items (mean={atk_c.mean():.3f})")
axes[1].set_title("Figure G.7b -- Item Confidence: Clean vs Attacked", fontsize=10)
axes[1].set_xlabel("Confidence Score"); axes[1].set_ylabel("Number of Items"); axes[1].legend()
plt.tight_layout(); save_eval("figG7_trust_confidence.png")
print("  Figure G.7 saved")

# G.8 Comprehensive dashboard
import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
x = np.arange(len(DS_ORDER)); w = 0.32

ax1 = fig.add_subplot(gs[0,0])
for i, model in enumerate(["CF_UU","SVD_MF"]):
    vals = [eval_table[(eval_table["Dataset"]==ds)&(eval_table["Model"]==model)]["RMSE"].values[0] for ds in DS_ORDER]
    ax1.bar(x+(i-0.5)*w, vals, w, color=DS_COLORS, alpha=0.9 if i==0 else 0.55,
            hatch="" if i==0 else "///", edgecolor="white", label=model)
ax1.set_title("RMSE by Dataset and Model", fontsize=10)
ax1.set_xticks(x); ax1.set_xticklabels(["Baseline","UNRR","Clean"], fontsize=8); ax1.legend(fontsize=7)

ax2 = fig.add_subplot(gs[0,1])
for i, model in enumerate(["CF_UU","SVD_MF"]):
    vals = [eval_table[(eval_table["Dataset"]==ds)&(eval_table["Model"]==model)]["NDCG@10"].values[0] for ds in DS_ORDER]
    ax2.bar(x+(i-0.5)*w, vals, w, color=DS_COLORS, alpha=0.9 if i==0 else 0.55,
            hatch="" if i==0 else "///", edgecolor="white", label=model)
ax2.set_title("NDCG@10 by Dataset and Model", fontsize=10)
ax2.set_xticks(x); ax2.set_xticklabels(["Baseline","UNRR","Clean"], fontsize=8); ax2.legend(fontsize=7)

ax3 = fig.add_subplot(gs[0,2])
for ds, color, ls in [("Baseline","#e05252","-"), ("UNRR","#2a9d8f","--")]:
    sub = stress_results[stress_results["Dataset"]==ds].sort_values("NoiseLevel")
    ax3.plot(sub["NoiseLevel"], sub["RMSE"], color=color, ls=ls, lw=2, marker="o", ms=5, label=ds)
ax3.set_title("RMSE vs Noise Level", fontsize=10)
ax3.set_xlabel("Contamination (%)"); ax3.set_ylabel("RMSE"); ax3.legend(fontsize=7)

ax4 = fig.add_subplot(gs[1,0])
bars = ax4.bar(range(4), ablation_df["RMSE"].values,
               color=["#e05252","#f4a261","#e9c46a","#2a9d8f"], edgecolor="white")
ax4.set_title("Ablation Study -- RMSE", fontsize=10)
ax4.set_xticks(range(4)); ax4.set_xticklabels(["Base","S1","S1+2","Full"], fontsize=8)
for bar, val in zip(bars, ablation_df["RMSE"].values):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
             f"{val:.4f}", ha="center", va="bottom", fontsize=7)

ax5 = fig.add_subplot(gs[1,1])
x2 = np.arange(len(atk_order)); w2 = 0.35
for i, (ds, color) in enumerate([("Baseline","#e05252"),("UNRR","#2a9d8f")]):
    vals = [attack_results[(attack_results["AttackType"]==a)&
                           (attack_results["Dataset"]==ds)]["RMSE"].values[0]
            if len(attack_results[(attack_results["AttackType"]==a)&
                                  (attack_results["Dataset"]==ds)]) else np.nan
            for a in atk_order]
    ax5.bar(x2+(i-0.5)*w2, vals, w2, label=ds, color=color, edgecolor="white", alpha=0.85)
ax5.set_title("RMSE by Attack Type", fontsize=10)
ax5.set_xticks(x2); ax5.set_xticklabels(atk_order, fontsize=8); ax5.legend(fontsize=7)

ax6 = fig.add_subplot(gs[1,2])
for _, row in eval_table.iterrows():
    mc = "#e05252" if row["Model"]=="CF_UU" else "#2a9d8f"
    mk = "o" if row["Dataset"]=="Injected" else ("s" if row["Dataset"]=="UNRR Filtered" else "^")
    ax6.scatter(row["R@10"], row["P@10"], color=mc, marker=mk, s=110, zorder=5)
    ax6.annotate(f"{row["Dataset"][:3]}", (row["R@10"], row["P@10"]),
                 textcoords="offset points", xytext=(3,3), fontsize=7)
ax6.set_title("Precision@10 vs Recall@10", fontsize=10)
ax6.set_xlabel("Recall@10"); ax6.set_ylabel("Precision@10")

plt.suptitle("Figure G.8 -- UNRR Framework: Comprehensive Evaluation Dashboard",
             fontsize=13, y=1.01, fontweight="bold")
save_eval("figG8_dashboard.png")
print("  Figure G.8 saved")
print("\n  All figures saved to ./eval_figures/")


  [Saved] eval_figures\figG7_trust_confidence.png
  Figure G.7 saved
  [Saved] eval_figures\figG8_dashboard.png
  Figure G.8 saved

  All figures saved to ./eval_figures/


## Step H -- Dependable Recommendations Output

Top-10 recommendations for sample users using the UNRR-cleaned SVD model.
Demonstrates the final output: dependable recommendations robust against both noise types.


In [51]:
def build_svd(train, k=50):
    """Sparse SVD-based matrix factorisation with k latent factors."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.linalg import svds
    gm  = train["Rating"].mean()
    u2i = {u: i for i, u in enumerate(train["UserID"].unique())}
    m2i = {m: i for i, m in enumerate(train["MovieID"].unique())}
    rows = [u2i[u] for u in train["UserID"]]
    cols = [m2i[m] for m in train["MovieID"]]
    vals = list(train["Rating"] - gm)
    mat  = csr_matrix((vals, (rows, cols)), shape=(len(u2i), len(m2i)))
    kk   = min(k, min(len(u2i), len(m2i)) - 1)
    U, s, Vt = svds(mat.astype(float), k=kk)
    pred_mat  = (U @ np.diag(s) @ Vt) + gm
    return {"U": U, "s": s, "Vt": Vt, "pred_mat": pred_mat,
            "u2i": u2i, "m2i": m2i, "gm": gm}

# Then call it normally
svd_final = build_svd(train_final, k=50)

In [54]:
# Redefine correct MovieLens SVD functions 
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

def build_svd(train, k=50):
    """Sparse SVD-based matrix factorisation with k latent factors."""
    gm  = train["Rating"].mean()
    u2i = {u: i for i, u in enumerate(train["UserID"].unique())}
    m2i = {m: i for i, m in enumerate(train["MovieID"].unique())}
    rows = [u2i[u] for u in train["UserID"]]
    cols = [m2i[m] for m in train["MovieID"]]
    vals = list(train["Rating"] - gm)
    mat  = csr_matrix((vals, (rows, cols)), shape=(len(u2i), len(m2i)))
    kk   = min(k, min(len(u2i), len(m2i)) - 1)
    U, s, Vt  = svds(mat.astype(float), k=kk)
    pred_mat  = (U @ np.diag(s) @ Vt) + gm
    return {"U": U, "s": s, "Vt": Vt, "pred_mat": pred_mat,
            "u2i": u2i, "m2i": m2i, "gm": gm}

def predict_svd(test, model):
    """Predict ratings using precomputed SVD decomposition."""
    pm, u2i, m2i, gm = (model["pred_mat"], model["u2i"],
                         model["m2i"],      model["gm"])
    preds = []
    for _, row in test.iterrows():
        uid, mid = row["UserID"], row["MovieID"]
        p = pm[u2i[uid], m2i[mid]] if (uid in u2i and mid in m2i) else gm
        preds.append(float(np.clip(p, 1, 5)))
    return np.array(preds)

print("✅ MovieLens SVD functions redefined")

✅ MovieLens SVD functions redefined


In [55]:
print("\n[H] DEPENDABLE RECOMMENDATIONS OUTPUT")
print("-" * 55)

train_final, _ = split_ratings(filtered_df, rating_col="AdjustedRating")
svd_final      = build_svd(train_final, k=50)
movies_meta    = pd.read_csv("movies.csv")[["MovieID","Title","Genres"]]

print("\n  Top-10 Recommendations for 5 sample users (UNRR-filtered SVD)")
print(f"  {'UserID':<10} {'Rank':<5} {'Movie Title':<45} {'Score':>6} {'Conf':>6}")
print("  " + "-" * 73)

sample_uids = genuine_df["UserID"].value_counts().head(5).index.tolist()
rec_records = []

for uid in sample_uids:
    rated_by_user = filtered_df[filtered_df["UserID"]==uid]["MovieID"].tolist()
    all_candidate = list(svd_final["m2i"].keys())
    unrated       = [m for m in all_candidate if m not in rated_by_user]
    if not unrated: continue

    test_rows = pd.DataFrame({"UserID": uid, "MovieID": unrated, "Rating": 0})
    preds     = predict_svd(test_rows, svd_final)
    top10_idx = np.argsort(preds)[-10:][::-1]
    top10_mid = [unrated[i] for i in top10_idx]
    top10_scr = preds[top10_idx]

    for rank, (mid, score) in enumerate(zip(top10_mid, top10_scr), 1):
        title = movies_meta[movies_meta["MovieID"]==mid]["Title"].values
        title = title[0] if len(title) else f"Movie {mid}"
        conf  = item_stats_inj[item_stats_inj["MovieID"]==mid]["Confidence"].values
        conf  = conf[0] if len(conf) else 1.0
        rec_records.append({
            "UserID": uid, "Rank": rank, "MovieID": mid,
            "Title": title, "PredScore": round(float(score),4),
            "ItemConfidence": round(float(conf),4)
        })
        if rank <= 3:
            print(f"  {uid:<10} {rank:<5} {title:<45} {score:>6.3f} {conf:>6.3f}")
    print(f"  {' '*10} ...   (showing top 3 of 10)")

recs_df = pd.DataFrame(rec_records)
recs_df.to_csv(
    os.path.join(INJECT_OUTPUT_DIR, "unrr_recommendations.csv"), index=False
)
print(f"\n  Full recommendations saved to {INJECT_OUTPUT_DIR}/unrr_recommendations.csv")


[H] DEPENDABLE RECOMMENDATIONS OUTPUT
-------------------------------------------------------

  Top-10 Recommendations for 5 sample users (UNRR-filtered SVD)
  UserID     Rank  Movie Title                                    Score   Conf
  -------------------------------------------------------------------------
  4169       1     Miss Julie (1999)                              4.228  0.501
  4169       2     Lost World: Jurassic Park, The (1997)          4.112  0.977
  4169       3     Alien (1979)                                   4.085  0.873
             ...   (showing top 3 of 10)
  1680       1     Monty Python and the Holy Grail (1974)         4.152  0.826
  1680       2     Pulp Fiction (1994)                            4.136  0.870
  1680       3     Stand by Me (1986)                             4.040  0.835
             ...   (showing top 3 of 10)
  1941       1     2001: A Space Odyssey (1968)                   4.363  0.841
  1941       2     Forrest Gump (1994)            

## Final Evaluation Summary Report

In [56]:
print("\n" + "=" * 65)
print("  UNRR FRAMEWORK -- FINAL EVALUATION SUMMARY REPORT")
print("=" * 65)

best_inj  = eval_table[eval_table["Dataset"]=="Injected"]["RMSE"].min()
best_flt  = eval_table[eval_table["Dataset"]=="UNRR Filtered"]["RMSE"].min()
best_gen  = eval_table[eval_table["Dataset"]=="Genuine"]["RMSE"].min()
best_p10  = eval_table[eval_table["Dataset"]=="UNRR Filtered"]["P@10"].max()
best_r10  = eval_table[eval_table["Dataset"]=="UNRR Filtered"]["R@10"].max()
best_ndcg = eval_table[eval_table["Dataset"]=="UNRR Filtered"]["NDCG@10"].max()
rmse_imp  = (best_inj - best_flt) / best_inj * 100

stress_d  = stress_results[stress_results["Dataset"]=="Baseline"].sort_values("NoiseLevel")
stress_u  = stress_results[stress_results["Dataset"]=="UNRR"].sort_values("NoiseLevel")
max_gap   = ((stress_d["RMSE"].values - stress_u["RMSE"].values) / stress_d["RMSE"].values * 100).max()
rmse_20d  = stress_d[stress_d["NoiseLevel"]==20.0]["RMSE"].values[0]
rmse_20u  = stress_u[stress_u["NoiseLevel"]==20.0]["RMSE"].values[0]

print(f"""
  PREDICTION ACCURACY (best model per dataset)
  Injected (Baseline) RMSE    : {best_inj:.4f}
  UNRR Filtered RMSE       : {best_flt:.4f}   <- Framework output
  Genuine (clean) RMSE     : {best_gen:.4f}   <- Upper bound
  RMSE Improvement         : {rmse_imp:+.2f}%

  RANKING QUALITY (UNRR Filtered, best model)
  Precision@10             : {best_p10:.4f}
  Recall@10                : {best_r10:.4f}
  NDCG@10                  : {best_ndcg:.4f}

  ROBUSTNESS UNDER HIGH NOISE (20% contamination)
  Baseline RMSE at 20%        : {rmse_20d:.4f}
  UNRR  RMSE at 20%        : {rmse_20u:.4f}
  Max RMSE gap             : {max_gap:.2f}%

  PIPELINE STATISTICS
  High-risk users removed  : {len(high_risk_uids):,}
  Genuine mean trust weight: {gen_tw:.4f}
  Malicious mean trust weight: {mal_tw:.4f}
  Mean item confidence     : {item_stats_inj["Confidence"].mean():.4f}
""")
print("=" * 65)
print("  UNRR Final Evaluation complete")
print("=" * 65)



  UNRR FRAMEWORK -- FINAL EVALUATION SUMMARY REPORT

  PREDICTION ACCURACY (best model per dataset)
  Injected (Baseline) RMSE    : 1.0570
  UNRR Filtered RMSE       : 0.9027   <- Framework output
  Genuine (clean) RMSE     : 1.0723   <- Upper bound
  RMSE Improvement         : +14.60%

  RANKING QUALITY (UNRR Filtered, best model)
  Precision@10             : 0.1942
  Recall@10                : 0.9663
  NDCG@10                  : 0.8057

  ROBUSTNESS UNDER HIGH NOISE (20% contamination)
  Baseline RMSE at 20%        : 1.1467
  UNRR  RMSE at 20%        : 1.0868
  Max RMSE gap             : 5.23%

  PIPELINE STATISTICS
  High-risk users removed  : 23
  Genuine mean trust weight: 0.8411
  Malicious mean trust weight: 0.8738
  Mean item confidence     : 0.8227

  UNRR Final Evaluation complete
